# Model 11: full-vector MLP evidence for new-animal detection

This is an experiment, not a claimed improvement. Attach the complete `projectwild3` competition data, including both supplied SIFT feature archives/directories. Use a CPU session; GPU is unused. Default settings run the full three-stage Dalvin baseline (12 episodes/stage, 400 candidate trees), then the MLP/calibration and three validation episodes. SIFT verification can take substantial time; output checkpoints/caches support resuming with identical settings.

Only supplied MIEW-ID vectors, SIFT descriptors/coordinates, metadata and labels are used. There are no raw images, pretrained networks, attention layers or deep-learning frameworks. The plain scikit-learn MLP learns symmetric interactions of the full 2,152-dimensional vectors. Dalvin's preserved baseline code is credited to Dalvin; the pair sampling design draws on the earlier Abhay Model 6 experiment, whose saturation failure motivates identity-based early stopping and query-level calibration here.

**Before looking at validation results:** the primary arm is a fixed 50% blend of baseline P(new) and the MLP-evidence calibrator. Candidate lists, known-only ordering, stacking and final ranking formula are shared. A summary-only calibration control tests whether adding the MLP gives useful information beyond recalibration. Full MLP replacement is diagnostic, not selected after validation.

The default seed reproduces an existing development allocation that has already been examined. Supervised training, calibration and validation identities are disjoint, but this is not a pristine final test or a leaderboard estimate. Episodes reuse photos/identities. A different seed requires fresh baseline training and a new output directory; do not repeatedly tune against validation results.

In [ ]:
from pathlib import Path
import sys, subprocess, json, importlib.metadata

MODE = 'train'  # 'train' for training/validation; 'submission' restores and skips training.
GENERATE_SUBMISSION = False  # Optional after training; submission mode always generates it.
RESTORE_ARCHIVE = None  # Optional Path('/kaggle/input/.../model11_checkpoint_bundle.zip')
BACKUP = Path('/kaggle/working/model11_checkpoint_bundle.zip')
SEED = 2052
WORKERS = 4
MLP_MAX_EPOCHS = 40
MAX_TRAIN_PAIRS = 24000
BASELINE_EPISODES_PER_STAGE = 12
CALIBRATION_EPISODES = 6
CANDIDATE_TREES = 400
OUTPUT = Path('/kaggle/working/model11_results')
DATA_ROOT = None  # Optional: set the folder containing train.csv and miewid_features.npy.
INSTALL_REQUIREMENTS = False  # Usually unnecessary on Kaggle; enable only if imports fail.
CODE = Path('/kaggle/working/model11_code')
CODE.mkdir(parents=True, exist_ok=True)


In [ ]:
import hashlib
SOURCES = {'dalvin_core.py': '"""Dalvin Model 3d functions; raw evidence can use a frozen nuisance projection."""\nimport os\nimport re\nimport cv2\nimport lightgbm as lgb\nimport numpy as np\nimport pandas as pd\nfrom multiprocessing import Pool\nfrom scipy import sparse\nfrom scipy.sparse.csgraph import connected_components\nfrom sklearn.decomposition import PCA\nfrom sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA\nK=15\nNEIGHBOURS=150\nMATCH_RATIO=.85\nRANSAC_PX=15.\nSIFT_PULL=.05\nVIEWS=[\'left\',\'right\',\'front\',\'back\',\'frontleft\',\'frontright\',\'backleft\',\'backright\',\'up\',\'down\']\nFLANK=np.array([0,1,2,2,0,1,0,1,2,2])\nREGIMES=[None,(.95,111),(.99,89),(.9,67)]\nPARAMS=dict(objective=\'binary\',learning_rate=.05,num_leaves=31,min_child_samples=30,subsample=.8,subsample_freq=1,colsample_bytree=.8,n_estimators=400,verbose=-1,random_state=42,n_jobs=4)\nINDEX_COLS=re.compile(r\'^(nb|nw|sb|sw)\\d$\')\n\n\n\n\n\n\n\ndef unit(a):\n    return a / np.linalg.norm(a, axis=1, keepdims=True)\n\n\ndef embed(gallery, labels, queries):\n    pca = PCA(min(384, gallery.shape[1], len(gallery) + len(queries) - 1), random_state=0).fit(np.r_[gallery, queries])      # unsupervised, sees the queries too\n    lda = LDA(solver="eigen", shrinkage=0.3).fit(pca.transform(gallery), labels)\n    project = lambda a: unit(lda.transform(pca.transform(a)))\n    return project(gallery), project(queries)\n\n\ndef per_identity(sim, inv, n):\n    """Per query: max and mean-of-top-2 similarity to each identity\'s gallery images."""\n    best, top2 = np.empty((sim.shape[0], n)), np.empty((sim.shape[0], n))\n    for j in range(n):\n        s = sim[:, inv == j]\n        best[:, j], top2[:, j] = s.max(1), np.sort(s, 1)[:, -2:].mean(1)\n    return best, top2\n\n\ndef species_features(gi, qi, sp):\n    """Query frame and (query, candidate) frame for one species; gi/qi are global image indices."""\n    Rg, Rq, yg, vg, vq = X[gi], X[qi], Y[gi], VIEW[gi], VIEW[qi]\n    Lg, Lq = embed(Rg, yg, Rq)\n    Rg, Rq = NX[gi], NX[qi]  # Frozen nuisance representation for raw evidence only.\n    ids, inv = np.unique(yg, return_inverse=True)\n    n, nq = len(ids), len(qi)\n    sim_l, sim_r = Lq @ Lg.T, Rq @ Rg.T\n    lda1, lda2 = per_identity(sim_l, inv, n)\n    raw1, raw2 = per_identity(sim_r, inv, n)\n    centroid = unit(np.stack([Lg[inv == j].mean(0) for j in range(n)]))\n    cen = Lq @ centroid.T\n    inl_g, good_g = INL[qi][:, gi].toarray().astype(np.float32), GOOD[qi][:, gi].toarray().astype(np.float32)\n    sift1, sift2 = per_identity(inl_g, inv, n)\n    good1, _ = per_identity(good_g, inv, n)\n    support = (inl_g >= 7).astype(np.float32) @ np.eye(n, dtype=np.float32)[inv]     # gallery images confirmed per identity\n    same_view = np.full((nq, n), np.nan)\n    for j in range(n):\n        for v in np.unique(vq):\n            cols = np.where((inv == j) & (vg == v))[0]\n            if len(cols):\n                same_view[vq == v, j] = sim_l[np.ix_(vq == v, cols)].max(1)\n    # viewpoint coverage: flanks of one animal look different, so unseen sides cannot be matched\n    by_view = np.stack([np.bincount(inv[vg == v], minlength=n) for v in range(len(VIEWS))], 1)\n    by_flank = np.stack([by_view[:, FLANK == f].sum(1) for f in range(3)], 1)\n    seen = vq >= 0\n    n_view = np.where(seen[:, None], by_view[:, vq.clip(0)].T, np.nan)\n    n_flank = np.where(seen[:, None], by_flank[:, FLANK[vq.clip(0)]].T, np.nan)\n    # transductive: neighbours among the queries themselves\n    qq_r, qq_l, qq_s = Rq @ Rq.T, Lq @ Lq.T, INL[qi][:, qi].toarray().astype(np.float32)\n    np.fill_diagonal(qq_r, -1), np.fill_diagonal(qq_l, -1)\n    nb, sb = np.argsort(-qq_l, 1)[:, :5], np.argsort(-qq_s, 1)[:, :5]\n    nbs, sbs = np.take_along_axis(qq_l, nb, 1), np.take_along_axis(qq_s, sb, 1)\n    t_l = np.take_along_axis(lda1, np.argsort(-lda1, 1)[:, :3], 1)\n    t_r = -np.sort(-raw1, 1)[:, :3]\n    t_s = -np.sort(-sift1, 1)[:, :3]\n    g_top = -np.sort(-sim_l, 1)\n    q = pd.DataFrame({\n        "sp": int(sp == "hyena"), "view": vq, "ng": len(gi),\n        "l1": t_l[:, 0], "l2": t_l[:, 1], "l3": t_l[:, 2], "r1": t_r[:, 0], "r2": t_r[:, 1], "r3": t_r[:, 2],\n        "lm12": t_l[:, 0] - t_l[:, 1], "rm12": t_r[:, 0] - t_r[:, 1],\n        "gd5": g_top[:, :5].mean(1), "gd20": g_top[:, :20].mean(1),\n        "qq1": nbs[:, 0], "qq3": nbs[:, :3].mean(1), "qq5": nbs.mean(1), "qqr1": qq_r.max(1),\n        "qn6": (qq_r > .6).sum(1), "qn7": (qq_r > .7).sum(1), "qn8": (qq_r > .8).sum(1),\n        "qq_minus_g": nbs[:, 0] - t_l[:, 0], "nb_g1": t_l[nb[:, 0], 0],\n        "s1": t_s[:, 0], "s2": t_s[:, 1], "s3": t_s[:, 2], "sg": good_g.max(1),\n        "qs1": sbs[:, 0], "qs3": sbs[:, :3].mean(1), "qsn10": (qq_s >= 10).sum(1), "qsn20": (qq_s >= 20).sum(1),\n        "qs_minus_s": sbs[:, 0] - t_s[:, 0], "nbs_s1": t_s[sb[:, 0], 0],\n    })\n    for k in range(5):\n        q[f"nb{k}"], q[f"nw{k}"], q[f"sb{k}"], q[f"sw{k}"] = nb[:, k], nbs[:, k], sb[:, k], sbs[:, k]\n    # candidates: best identities by embedding similarity, lightly boosted by SIFT evidence\n    cand = np.argsort(-(lda1 + SIFT_PULL * np.tanh(sift1 / 15)), 1)[:, :K]\n    r, c = np.repeat(np.arange(nq), K), cand.ravel()\n    nb_ev = np.max(np.minimum(nbs[:, :, None], np.stack([lda1[nb[:, k]] for k in range(5)], 1)), 1)\n    sb_ev = np.max(np.minimum(sbs[:, :, None], np.stack([sift1[sb[:, k]] for k in range(5)], 1)), 1)\n    rank_of = lambda m: (-m).argsort(1).argsort(1)[r, c]\n    cd = pd.DataFrame({\n        "q": r, "c": c, "rank": np.tile(np.arange(K), nq), "ident": ids[c],\n        "mxr": raw1[r, c], "m2r": raw2[r, c], "mxl": lda1[r, c], "m2l": lda2[r, c], "cl": cen[r, c],\n        "gapl": lda1[r, c] - t_l[r, 0], "gapr": raw1[r, c] - t_r[r, 0], "same": same_view[r, c],\n        "cnt": np.bincount(inv)[c], "nbev": nb_ev[r, c], "sbev": sb_ev[r, c],\n        "rk_r": rank_of(raw1), "rk_c": rank_of(cen), "rk_s": rank_of(sift1),\n        "n_view": n_view[r, c], "n_flank": n_flank[r, c], "support": support[r, c], "sx": sift1[r, c], "s2x": sift2[r, c], "gx": good1[r, c], "sgap": sift1[r, c] - t_s[r, 0],\n    })\n    return q, cd\n\n\ndef build_frames(gi, qi):\n    qs, cs, order, off = [], [], [], 0\n    for sp in ("hyena", "leopard"):\n        g, qq = gi[SP[gi] == sp], qi[SP[qi] == sp]\n        q, c = species_features(g, qq, sp)\n        c["q"] += off\n        q[[f"{p}{k}" for p in ("nb", "sb") for k in range(5)]] += off\n        qs.append(q), cs.append(c), order.append(qq)\n        off += len(qq)\n    return dict(q=pd.concat(qs, ignore_index=True), c=pd.concat(cs, ignore_index=True), order=np.concatenate(order))\n\n\ndef duplicate_groups(ix, link):\n    """Group images of one identity that are near-duplicates (same encounter); each image alone if link is None."""\n    if link is None:\n        return np.arange(len(ix))\n    edges = (X[ix] @ X[ix].T > link[0]) | (INL[ix][:, ix].toarray() >= link[1])\n    return connected_components(sparse.csr_matrix(edges))[1]\n\n\ndef columns(s):\n    qf = [c for c in s["q"].columns if c != "unk" and not INDEX_COLS.match(c)]\n    cf = [c for c in s["c"].columns if c not in ("q", "c", "ident", "y")]\n    return qf, cf\n\n\ndef candidate_matrix(s, qf, cf):\n    return pd.concat([s["c"][cf].reset_index(drop=True), s["q"].iloc[s["c"].q.to_numpy()][qf].reset_index(drop=True)], axis=1)\n\n\ndef fit(sims):\n    """new-individual detector, closed-set candidate scorer, and all-query candidate scorer."""\n    qf, cf = columns(sims[0])\n    known = lambda s: s["q"].unk.to_numpy()[s["c"].q.to_numpy()] == 0\n    cx = pd.concat([candidate_matrix(s, qf, cf) for s in sims])\n    cy = np.concatenate([s["c"].y.to_numpy() for s in sims])\n    ck = np.concatenate([known(s) for s in sims])\n    detector = lgb.LGBMClassifier(**{**PARAMS, "n_estimators": 300, "num_leaves": 15}).fit(\n        pd.concat([s["q"][qf] for s in sims]), np.concatenate([s["q"].unk.to_numpy() for s in sims]))\n    return detector, lgb.LGBMClassifier(**PARAMS).fit(cx[ck], cy[ck]), lgb.LGBMClassifier(**PARAMS).fit(cx, cy), (qf, cf)\n\n\ndef predict(models, s):\n    detector, closed, anyq, (qf, cf) = models\n    nq, cx = len(s["q"]), candidate_matrix(s, qf, cf)\n    p_new = detector.predict_proba(s["q"][qf])[:, 1]\n    raw = anyq.predict_proba(cx)[:, 1].reshape(nq, K)\n    within = closed.predict_proba(cx)[:, 1].reshape(nq, K)\n    within /= np.maximum(within.sum(1, keepdims=True), 1e-9)\n    return dict(pn=p_new, raw=raw, pc=within, pb=(p_new + np.clip(1 - raw.sum(1), 0, 1)) / 2)\n\n\ndef rank(s, r):\n    """Top-5 labels: P(new) vs (1 - P(new)) * P(identity | known)."""\n    nq = len(s["q"])\n    labels = np.concatenate([s["c"].ident.to_numpy(dtype=object).reshape(nq, K), np.full((nq, 1), "new_individual", dtype=object)], 1)\n    scores = np.concatenate([(1 - r["pb"])[:, None] * r["pc"], r["pb"][:, None]], 1)\n    return np.take_along_axis(labels, np.argsort(-scores, 1)[:, :5], 1)\n\n\ndef map5(preds, truth):\n    return np.mean([next((1 / (i + 1) for i, p in enumerate(row) if p == t), 0.0) for row, t in zip(preds, truth)])\n\n\ndef augment(s, r):\n    """Add the previous round\'s predictions for each query and for its nearest query neighbours."""\n    q, c, nq = s["q"].copy(), s["c"].copy(), len(s["q"])\n    codes = pd.factorize(c.ident)[0].reshape(nq, K)\n    q["r_pn"], q["r_pb"], q["r_rawsum"], q["r_rawmax"] = r["pn"], r["pb"], r["raw"].sum(1), r["raw"].max(1)\n    c["r_raw"], c["r_pc"] = r["raw"].ravel(), r["pc"].ravel()\n    for kind, weight, scale in (("nb", "nw", 1.0), ("sb", "sw", 30.0)):     # embedding / SIFT neighbours\n        evidence, support, pb, w, top = np.zeros((nq, K)), np.zeros((nq, K)), [], [], []\n        for k in range(5):\n            idx, sim = q[f"{kind}{k}"].to_numpy(), q[f"{weight}{k}"].to_numpy() / scale\n            wk = np.clip(sim - 0.4, 0, None) if kind == "nb" else np.clip(sim, 0, 1)\n            agree = ((codes[idx][:, None, :] == codes[:, :, None]) * r["raw"][idx][:, None, :]).sum(2)\n            evidence = np.maximum(evidence, np.minimum(agree, np.clip(sim, 0, 1)[:, None]))\n            support += wk[:, None] * agree\n            pb.append(r["pb"][idx]), w.append(wk), top.append(r["raw"][idx].max(1))\n        pb, w = np.stack(pb, 1), np.stack(w, 1)\n        q[f"{kind}_pb0"], q[f"{kind}_pbw"] = pb[:, 0], (pb * w).sum(1) / np.maximum(w.sum(1), 1e-6)\n        q[f"{kind}_pbmin"], q[f"{kind}_w"], q[f"{kind}_rm0"] = np.where(w > 0, pb, 1).min(1), w.sum(1), top[0]\n        c[f"r_{kind}ev"], c[f"r_{kind}sum"] = evidence.ravel(), support.ravel()\n    return dict(s, q=q, c=c)\n', 'data_protocol.py': '"""Data loading, role-local mutual SIFT and identity-disjoint episode helpers."""\nimport os\nimport hashlib\nfrom pathlib import Path\nfrom concurrent.futures import ThreadPoolExecutor\nfrom functools import lru_cache\nfrom threading import Lock\nimport numpy as np\nimport pandas as pd\nimport cv2\nfrom scipy import sparse\nimport dalvin_core as b\nNEW=\'new_individual\'\ndef find_sift(root, split):\n    """Kaggle mounts SIFT beside the metadata folder, sometimes as extracted NPZ directories."""\n    name = f"sift_features_{split}.npz"\n    for parent in (root, root.parent, root.parent.parent):\n        candidate = parent / name\n        if candidate.exists():\n            return candidate\n    raise FileNotFoundError(f"Cannot find {name} in {root} or its two parent directories. "\n                            "Attach the full projectwild3 competition input, including both SIFT folders.")\n\n\nclass SiftSource:\n    def __init__(self, path):\n        self.archive = None\n        self.files = {}\n        if path.is_file():\n            self.archive = np.load(path)\n        else:\n            for file in path.rglob("*.npy"):\n                if file.stem in self.files:\n                    raise ValueError(f"Duplicate SIFT key {file.stem} in {path}")\n                self.files[file.stem] = file\n            if not self.files:\n                raise ValueError(f"No extracted .npy SIFT arrays in {path}")\n\n    def __getitem__(self, key):\n        return self.archive[key] if self.archive is not None else np.load(self.files[key], allow_pickle=False)\n\n\ndef load_data(root):\n    train = pd.read_csv(root / "train.csv")\n    test = pd.read_csv(root / "test.csv")\n    ids = pd.read_csv(root / "miewid_ids.csv")\n    all_df = pd.concat([train, test.assign(identity=None)], ignore_index=True)\n    if not all_df.id.is_unique or not ids.id.is_unique:\n        raise ValueError("Duplicate image IDs")\n    for col in ("species", "viewpoint", "identity"):\n        if col not in all_df:\n            raise ValueError(f"Missing column: {col}")\n    if train.identity.isna().any() or not set(all_df.species) <= {"hyena", "leopard"}:\n        raise ValueError("Invalid identity/species metadata")\n    raw = np.load(root / "miewid_features.npy", mmap_mode="r")\n    if len(raw) != len(ids) or set(ids.id) != set(all_df.id):\n        raise ValueError("Feature ID mapping does not match metadata")\n    ix = pd.Index(ids.id).get_indexer(all_df.id)\n    X = np.asarray(raw[ix], dtype=np.float32)\n    if not np.isfinite(X).all() or np.any(np.linalg.norm(X, axis=1) == 0):\n        raise ValueError("Features contain nonfinite values or zero vectors")\n    b.X = b.unit(X)\n    b.IDS = all_df.id.to_numpy()\n    b.Y = all_df.identity.to_numpy(dtype=object)\n    b.SP = all_df.species.to_numpy()\n    b.VIEW = all_df.viewpoint.map({v: i for i, v in enumerate(b.VIEWS)}).fillna(-1).to_numpy(int)\n    return train, test, all_df\n\n\nclass SiftCache:\n    """Role-local neighbour selection; thread workers share lazily loaded descriptors."""\n    def __init__(self, root, out, workers, neighbours):\n        paths = [find_sift(root, s) for s in ("train", "test")]\n        print("SIFT sources:", *paths, sep="\\n", flush=True)\n        self.z = [SiftSource(path) for path in paths]\n        self.read_lock = Lock()\n        self.workers, self.neighbours = workers, neighbours\n        fingerprint = hashlib.sha256(b.IDS.tobytes() if b.IDS.dtype != object else "\\n".join(b.IDS).encode())\n        fingerprint.update(np.ascontiguousarray(b.X).tobytes())\n        self.path = out / ("sift_mutual_" + fingerprint.hexdigest()[:16] + ".npz")\n        self.done = {}\n        if self.path.exists():\n            with np.load(self.path) as z:\n                self.done = {tuple(p): tuple(v) for p, v in zip(z["pairs"], z["values"])}\n\n    @lru_cache(maxsize=1200)\n    def descriptors(self, i):\n        iid = b.IDS[i]\n        z = self.z[0 if iid.startswith("tr_") else 1]\n        # NpzFile\'s shared ZIP handle is not safe for concurrent reads.\n        with self.read_lock:\n            d = np.asarray(z[f"{iid}_desc"], dtype=np.float32)\n            xy = np.asarray(z[f"{iid}_xy"], dtype=np.float32)\n        if d.ndim != 2 or xy.shape != (len(d), 2):\n            raise ValueError(f"Malformed SIFT arrays for {iid}")\n        d = np.sqrt(d / np.maximum(d.sum(1, keepdims=True), 1))\n        return d, xy\n\n    def verify(self, pair):\n        a, c = pair\n        da, xa = self.descriptors(a)\n        dc, xc = self.descriptors(c)\n        if min(len(da), len(dc)) < 4:\n            return 0, 0\n        bf = cv2.BFMatcher()\n        back = np.array([m.trainIdx for m in bf.match(dc, da)])\n        matches = bf.knnMatch(da, dc, k=2)\n        good = [m[0] for m in matches if len(m) == 2 and m[0].distance < .85*m[1].distance\n                and back[m[0].trainIdx] == m[0].queryIdx]\n        if len(good) < 4:\n            return len(good), 0\n        # Local RNG seed per task. OpenCV RNG state is thread-local.\n        cv2.setRNGSeed(int((int(a)*100003 + int(c)) % (2**31-1)))\n        _, mask = cv2.estimateAffinePartial2D(\n            np.float32([xa[m.queryIdx] for m in good]),\n            np.float32([xc[m.trainIdx] for m in good]),\n            method=cv2.RANSAC, ransacReprojThreshold=15.)\n        return len(good), int(mask.sum()) if mask is not None else 0\n\n    def activate(self, indices):\n        wanted = set()\n        for sp in ("hyena", "leopard"):\n            ix = indices[b.SP[indices] == sp]\n            # Chunked similarities avoid allocating the full role x role matrix.\n            for start in range(0, len(ix), 256):\n                q = ix[start:start+256]\n                s = b.X[q] @ b.X[ix].T\n                s[np.arange(len(q)), np.arange(start, start+len(q))] = -np.inf\n                k = min(self.neighbours, len(ix)-1)\n                nb = np.argsort(-s, axis=1)[:, :k]\n                for a, row in zip(q, nb):\n                    wanted.update((min(int(a), int(c)), max(int(a), int(c))) for c in ix[row])\n        missing = sorted(wanted - self.done.keys())\n        print(f"SIFT: {len(wanted)} role-local pairs; {len(missing)} uncached", flush=True)\n        with ThreadPoolExecutor(max_workers=self.workers) as pool:\n            for j, (pair, value) in enumerate(zip(missing, pool.map(self.verify, missing)), 1):\n                self.done[pair] = value\n                if j % 5000 == 0:\n                    self.save()\n                    print(f"  verified {j}/{len(missing)}", flush=True)\n        self.save()\n        # Only expose requested pairs, even when cache contains other roles/test.\n        pp = np.array(sorted(wanted), dtype=int).reshape(-1, 2)\n        vv = np.array([self.done[tuple(p)] for p in pp]).reshape(-1, 2)\n        n = len(b.X)\n        def matrix(v):\n            a, c = pp.T\n            return sparse.coo_matrix((np.r_[v, v], (np.r_[a, c], np.r_[c, a])), shape=(n, n)).tocsr()\n        b.GOOD, b.INL = matrix(vv[:, 0]), matrix(vv[:, 1])\n\n    def save(self):\n        """Persist completed verifications atomically so interrupted runs can resume."""\n        pairs = sorted(self.done)\n        tmp = self.path.with_suffix(".tmp.npz")\n        np.savez_compressed(tmp, pairs=np.array(pairs, dtype=np.int64).reshape(-1, 2),\n                            values=np.array([self.done[p] for p in pairs]).reshape(-1, 2))\n        os.replace(tmp, self.path)\n\n\ndef allocate_roles(train, rounds, seed):\n    rng = np.random.default_rng(seed)\n    roles = {f"stage{i+1}": [] for i in range(rounds)} | {"calibration": [], "validation": []}\n    for sp in ("hyena", "leopard"):\n        names = np.array(sorted(train.loc[train.species == sp, "identity"].unique()))\n        rng.shuffle(names)\n        n = max(1, int(round(.2*len(names))))\n        roles["validation"].extend(names[:n])\n        roles["calibration"].extend(names[n:2*n])\n        for i, block in enumerate(np.array_split(names[2*n:], rounds)):\n            roles[f"stage{i+1}"].extend(block)\n    sets = [set(v) for v in roles.values()]\n    assert all(not a & c for i, a in enumerate(sets) for c in sets[i+1:])\n    return {k: np.flatnonzero(train.identity.isin(v).to_numpy()) for k, v in roles.items()}\n\n\ndef split_episode(indices, seed, cap=32, link=(.95, 100)):\n    """Unknown identities absent from gallery; known-query duplicate groups kept whole."""\n    rng = np.random.default_rng(seed)\n    gallery, known, unknown = [], [], []\n    for sp in ("hyena", "leopard"):\n        ix = indices[b.SP[indices] == sp]\n        names = np.unique(b.Y[ix])\n        rng.shuffle(names)\n        nu = max(1, int(round(rng.uniform(.2, .35)*len(names))))\n        unseen = set(names[:nu])\n        for name in names:\n            rows = ix[b.Y[ix] == name].copy()\n            rng.shuffle(rows)\n            if name in unseen:\n                unknown.extend(rows[:cap])\n                continue\n            held = []\n            groups = b.duplicate_groups(rows, link)\n            target = min(max(1, int(round(.35*len(rows)))), cap)\n            for c in rng.permutation(np.unique(groups)):\n                members = rows[groups == c].tolist()\n                if len(held) < target and len(held)+len(members) < len(rows):\n                    held.extend(members)\n            # Never truncate a duplicate group after selecting it.\n            known.extend(held)\n            gallery.extend(i for i in rows if i not in set(held))\n    gi, qi = np.array(gallery, dtype=int), np.array(known+unknown, dtype=int)\n    if not len(known) or not len(unknown):\n        raise ValueError("Episode lacks known/unknown queries; use a larger role or different seed")\n    for sp in ("hyena", "leopard"):\n        if len(np.unique(b.Y[gi[b.SP[gi] == sp]])) < b.K or np.sum(b.SP[qi] == sp) < 6:\n            raise ValueError("Too few identities/queries per species; reduce --rounds or --candidates")\n    assert not set(b.Y[unknown]) & set(b.Y[gi])\n    assert set(b.Y[known]) <= set(b.Y[gi])\n    return gi, qi, np.where(np.isin(qi, unknown), NEW, b.Y[qi])\n\n\ndef frame(gi, qi, truth=None):\n    s = b.build_frames(gi, qi)\n    if truth is not None:\n        lookup = dict(zip(qi, truth))\n        s["truth"] = np.array([lookup[i] for i in s["order"]])\n        s["q"]["unk"] = (s["truth"] == NEW).astype(int)\n        s["c"]["y"] = (s["c"].ident.to_numpy() == s["truth"][s["c"].q.to_numpy()]).astype(int)\n    return s\n\n\ndef through_stack(s, models):\n    for i, m in enumerate(models):\n        r = b.predict(m, s)\n        if i < len(models)-1:\n            s = b.augment(s, r)\n    return s, r\n', 'verifier.py': '"""Full-vector pair MLP and gallery-aware unknown calibration. No raw images."""\nimport copy\nimport numpy as np\nfrom scipy.sparse import csr_matrix\nfrom scipy.sparse.csgraph import connected_components\nfrom sklearn.neural_network import MLPClassifier\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.metrics import log_loss, roc_auc_score\n\n\ndef pair_features(x, a, c):\n    """Symmetric, full 2152-D descriptor interactions, without identity inputs."""\n    return np.c_[np.abs(x[a]-x[c]),x[a]*x[c],np.sum(x[a]*x[c],axis=1)].astype(np.float32)\n\n\ndef sample_pairs(x,y,sp,indices,seed,max_pairs=24000):\n    rng=np.random.default_rng(seed); pairs=[]; labels=[]\n    for species in (\'hyena\',\'leopard\'):\n        ix=indices[sp[indices]==species]; sim=x[ix]@x[ix].T\n        groups=np.zeros(len(ix),int); offset=0\n        for name in np.unique(y[ix]):\n            loc=np.flatnonzero(y[ix]==name)\n            _,gg=connected_components(csr_matrix(sim[np.ix_(loc,loc)]>=.97),directed=False)\n            groups[loc]=gg+offset; offset+=gg.max()+1\n        # Cap anchor appearances per identity so large identities do not dominate.\n        anchors=[]\n        for name in np.unique(y[ix]):\n            loc=np.flatnonzero(y[ix]==name)\n            anchors.extend(rng.choice(loc,min(24,len(loc)),replace=False))\n        for j in anchors:\n            pos=np.flatnonzero((y[ix]==y[ix[j]])&(groups!=groups[j]))\n            neg=np.flatnonzero(y[ix]!=y[ix[j]])\n            if not len(pos) or not len(neg):continue\n            hard=neg[np.argsort(-sim[j,neg])[:30]]\n            for _ in range(2):\n                for candidate,label in ((rng.choice(pos),1),(rng.choice(hard),0),(rng.choice(neg),0)):\n                    pairs.append((ix[j],ix[candidate])); labels.append(label)\n    if not pairs:raise ValueError(\'No usable training pairs\')\n    pairs=np.asarray(pairs); labels=np.asarray(labels)\n    # Deduplicate unordered pairs before sampling to avoid duplicate weights.\n    pairs=np.sort(pairs,axis=1); _,keep=np.unique(pairs,axis=0,return_index=True)\n    keep=rng.permutation(keep)[:max_pairs]\n    if len(np.unique(labels[keep]))!=2:raise ValueError(\'Pair classes missing\')\n    return pairs[keep],labels[keep]\n\n\ndef fit_verifier(x,y,sp,indices,seed=2052,epochs=40,max_pairs=24000):\n    rng=np.random.default_rng(seed); fit=[]; stop=[]\n    for species in (\'hyena\',\'leopard\'):\n        names=np.unique(y[indices[sp[indices]==species]]); rng.shuffle(names)\n        n=max(2,round(.2*len(names)))\n        stop.extend(indices[(sp[indices]==species)&np.isin(y[indices],names[:n])])\n        fit.extend(indices[(sp[indices]==species)&np.isin(y[indices],names[n:])])\n    fit=np.asarray(fit); stop=np.asarray(stop)\n    assert not set(y[fit]) & set(y[stop])\n    fp,fy=sample_pairs(x,y,sp,fit,seed,max_pairs)\n    ep,ey=sample_pairs(x,y,sp,stop,seed+1,min(8000,max_pairs))\n    scaler=StandardScaler().fit(pair_features(x,*fp.T))\n    transform=lambda pairs:np.clip(scaler.transform(pair_features(x,*pairs.T)),-6,6).astype(np.float32)\n    xx,ee=transform(fp),transform(ep)\n    net=MLPClassifier(hidden_layer_sizes=(32,16),alpha=1.,batch_size=256,\n        learning_rate_init=.0003,random_state=seed,early_stopping=False)\n    best=None; best_loss=np.inf; stale=0; history=[]\n    for epoch in range(1,epochs+1):\n        order=rng.permutation(len(fy)); net.partial_fit(xx[order],fy[order],classes=[0,1])\n        pr=net.predict_proba(ee)[:,1]; loss=log_loss(ey,pr,labels=[0,1])\n        history.append(dict(epoch=epoch,stop_logloss=float(loss),stop_auc=float(roc_auc_score(ey,pr))))\n        print(\'MLP epoch\',epoch,\'identity-stop loss\',round(loss,5),flush=True)\n        if loss<best_loss-1e-4:best_loss=loss; best=copy.deepcopy(net); stale=0\n        else:stale+=1\n        if stale>=6:break\n    parameters=sum(v.size for v in best.coefs_+best.intercepts_)\n    return dict(network=best,scaler=scaler,history=history,parameters=parameters,\n                fit_indices=fit,stop_indices=stop,fit_pairs=len(fy),stop_pairs=len(ey))\n\n\ndef pair_scores(model,x,a,c,batch=512):\n    result=[]\n    for start in range(0,len(a),batch):\n        f=pair_features(x,a[start:start+batch],c[start:start+batch])\n        z=np.clip(model[\'scaler\'].transform(f),-6,6)\n        result.append(model[\'network\'].predict_proba(z)[:,1])\n    return np.concatenate(result) if result else np.array([])\n\n\ndef query_features(model,x,y,gallery,s,r):\n    """No query truth is used. Score every gallery photo of all 15 candidates."""\n    n=len(s[\'q\']); k=15; ids=s[\'c\'].ident.to_numpy().reshape(n,k)\n    blocks={name:gallery[y[gallery]==name] for name in np.unique(ids)}\n    aa=[]; cc=[]; sizes=[]\n    for i,query in enumerate(s[\'order\']):\n        for name in ids[i]:\n            g=blocks[name]; aa.extend([query]*len(g)); cc.extend(g); sizes.append(len(g))\n    pp=pair_scores(model,x,np.asarray(aa),np.asarray(cc)); statistics=[]; offset=0\n    for size in sizes:\n        v=np.sort(pp[offset:offset+size])[::-1]; offset+=size\n        statistics.append([v[0],v[:2].mean(),v[:3].mean(),v.mean(),(v>.5).mean()])\n    statistics=np.asarray(statistics).reshape(n,k,5)\n    count=np.asarray(sizes).reshape(n,k); top=r[\'pc\'].argmax(1); ix=np.arange(n)\n    get=lambda key:s[\'c\'][key].to_numpy().reshape(n,k)[ix,top]\n    sorted_pc=np.sort(r[\'pc\'],axis=1)\n    # The control and MLP arm have identical baseline context and calibration model.\n    context=np.c_[r[\'pn\'],r[\'pb\'],np.clip(1-r[\'raw\'].sum(1),0,1),sorted_pc[:,-1],sorted_pc[:,-2],\n        -(r[\'pc\']*np.log(np.maximum(r[\'pc\'],1e-12))).sum(1),np.log1p(count[ix,top]),\n        get(\'mxr\'),get(\'mxl\'),np.log1p(get(\'sx\')),np.log1p(get(\'n_flank\')),\n        np.log1p(get(\'n_view\')),s[\'q\'].sp.to_numpy(),s[\'q\'].view.to_numpy()<0]\n    sorted_best=np.sort(statistics[:,:,0],axis=1)\n    richest=statistics[:,:,0].argmax(1)\n    extra=np.c_[statistics[ix,top],statistics.max(1),\n        (statistics*r[\'pc\'][:,:,None]).sum(1),sorted_best[:,-1]-sorted_best[:,-2],\n        np.log1p(count[ix,richest]),np.log1p(count.sum(1))]\n    return np.nan_to_num(context),np.nan_to_num(np.c_[context,extra])\n\n\ndef calibrator(features,unknown):\n    # Query probabilities are learned on separate calibration identities;\n    # pair scores themselves are never treated as calibrated P(known).\n    return make_pipeline(StandardScaler(),LogisticRegression(C=.1,max_iter=1000)).fit(features,unknown)\n\n\ndef rank(candidates,pc,pnew):\n    labels=np.c_[candidates,np.full(len(pc),\'new_individual\',dtype=object)]\n    scores=np.c_[(1-pnew[:,None])*pc,pnew]\n    return np.take_along_axis(labels,np.argsort(-scores,axis=1)[:,:5],axis=1)\n\n\ndef rr5(pred,truth):\n    return np.array([next((1/(i+1) for i,label in enumerate(row) if label==target),0.) for row,target in zip(pred,truth)])\n', 'run.py': '"""Paired open-set evaluation: full-vector MLP evidence for P(new)."""\nimport os\nos.environ.setdefault(\'OMP_NUM_THREADS\',\'4\'); os.environ.setdefault(\'OPENBLAS_NUM_THREADS\',\'4\')\nimport sys\nfrom pathlib import Path\nHERE=Path(__file__).resolve().parent\nsys.path.insert(0,str(HERE.parents[2]/\'tmp/abhay_deps\'))\nimport argparse,json,hashlib,warnings\nimport numpy as np\nimport pandas as pd\nimport cv2,joblib\nfrom threadpoolctl import threadpool_limits\nimport dalvin_core as b\nimport data_protocol as p\nimport verifier as v\n\n\ndef metrics(d,column):\n    return dict(map5=float(d[column].mean()),known_map5=float(d.loc[~d.unknown,column].mean()),\n        unknown_map5=float(d.loc[d.unknown,column].mean()),\n        unknown_top1=float((d.loc[d.unknown,column]==1).mean()),\n        known_top1=float((d.loc[~d.unknown,column]==1).mean()))\n\n\ndef bootstrap(d,column):\n    z=d.assign(delta=d[column]-d.baseline_rr5).groupby([\'species\',\'identity\']).delta.agg([\'sum\',\'size\'])\n    rng=np.random.default_rng(2052); sums=np.zeros(2000); counts=np.zeros(2000)\n    for _,part in z.groupby(level=0):\n        a=part.to_numpy(); ix=rng.integers(0,len(a),(2000,len(a)))\n        sums+=a[ix,0].sum(1); counts+=a[ix,1].sum(1)\n    return np.quantile(sums/counts,[.025,.975]).tolist()\n\n\ndef main():\n    ap=argparse.ArgumentParser(); ap.add_argument(\'--data\',type=Path,required=True)\n    ap.add_argument(\'--output\',type=Path,default=HERE/\'local_runs/main\')\n    ap.add_argument(\'--seed\',type=int,default=2052); ap.add_argument(\'--workers\',type=int,default=4)\n    ap.add_argument(\'--epochs\',type=int,default=40); ap.add_argument(\'--max-pairs\',type=int,default=24000)\n    ap.add_argument(\'--baseline-sims\',type=int,default=12); ap.add_argument(\'--calibration-episodes\',type=int,default=6)\n    ap.add_argument(\'--trees\',type=int,default=400); ap.add_argument(\'--reuse-local\',action=\'store_true\')\n    args=ap.parse_args(); out=args.output; out.mkdir(parents=True,exist_ok=True)\n    if min(args.epochs,args.max_pairs,args.baseline_sims,args.calibration_episodes,args.trees,args.workers)<1:raise ValueError(\'Counts must be positive\')\n    config={k:str(a) if isinstance(a,Path) else a for k,a in vars(args).items()}\n    config.update(primary=\'mlp_blend\',blend_weight=.5,roles=\'60% baseline/verifier,20% calibration,20% validation\',\n        pair_stop=\'20% of verifier training identities, disjoint from pair fitting\',\n        calibrated_label=\'query animal absent from episode gallery\',\n        sources={name:hashlib.sha256((HERE/name).read_bytes()).hexdigest() for name in [\'run.py\',\'verifier.py\',\'dalvin_core.py\',\'data_protocol.py\']})\n    path=out/\'config.json\'\n    if path.exists() and json.loads(path.read_text())!=config:raise ValueError(\'Configuration/source changed: choose a NEW output directory\')\n    path.write_text(json.dumps(config,indent=2))\n    cv2.setNumThreads(1);warnings.filterwarnings(\'ignore\',message=\'Only one sample available.*\',category=UserWarning)\n    b.PARAMS.update(n_estimators=args.trees,n_jobs=args.workers,random_state=args.seed)\n    train,_,_=p.load_data(args.data); b.NX=b.X\n    roles=p.allocate_roles(train,3,args.seed); supervised=np.concatenate([roles[f\'stage{i}\'] for i in (1,2,3)])\n    assert not set(b.Y[supervised])&set(b.Y[roles[\'calibration\']])\n    assert not set(b.Y[roles[\'validation\']])&set(b.Y[np.r_[supervised,roles[\'calibration\']]])\n    pd.DataFrame([dict(id=b.IDS[i],identity=b.Y[i],species=b.SP[i],role=role) for role,ix in roles.items() for i in ix]).to_csv(out/\'identity_roles.csv\',index=False)\n    cache=p.SiftCache(args.data,out,args.workers,150)\n    source=HERE.parent/\'Model7/local_runs/paired_2052\'\n    if args.reuse_local:\n        if args.seed!=2052 or args.trees!=400 or args.baseline_sims!=12:raise ValueError(\'Local baseline reuse requires original settings\')\n        for directory in [source,HERE.parent/\'Model9/local_runs/coverage_2052\']:\n            for file in directory.glob(\'sift_mutual_*.npz\'):\n                with np.load(file) as z:cache.done.update({tuple(k):tuple(val) for k,val in zip(z[\'pairs\'],z[\'values\'])})\n    baseline_path=out/\'baseline_models.joblib\'\n    if baseline_path.exists():models=joblib.load(baseline_path)\n    elif args.reuse_local:\n        models=joblib.load(source/\'paired_models.joblib\')[\'baseline\'];joblib.dump(models,baseline_path)\n    else:models=[]\n    splitrows=[]\n    def episode(indices,seed,role,link):\n        gi,qi,truth=p.split_episode(indices,seed,link=link)\n        splitrows.extend(dict(episode=seed,role=role,id=b.IDS[i],assignment=\'gallery\') for i in gi)\n        splitrows.extend(dict(episode=seed,role=role,id=b.IDS[i],assignment=\'unknown_query\' if t==p.NEW else \'known_query\') for i,t in zip(qi,truth))\n        return gi,qi,truth\n    for stage in range(3):\n        if stage<len(models):continue\n        ix=roles[f\'stage{stage+1}\'];cache.activate(ix);frames=[]\n        for j in range(args.baseline_sims):\n            gi,qi,truth=episode(ix,args.seed+1000*(stage+1)+j,f\'stage{stage+1}\',b.REGIMES[j%4])\n            s=p.frame(gi,qi,truth)\n            for model in models:s=b.augment(s,b.predict(model,s))\n            frames.append(s);print(\'Baseline stage\',stage+1,\'episode\',j+1,flush=True)\n        models.append(b.fit(frames));joblib.dump(models,baseline_path)\n    network_path=out/\'verifier.joblib\'\n    if network_path.exists():network=joblib.load(network_path)\n    else:\n        network=v.fit_verifier(b.X,b.Y,b.SP,supervised,args.seed,args.epochs,args.max_pairs)\n        joblib.dump(network,network_path)\n    pairroles={i:\'pair_fit\' for i in network[\'fit_indices\']}|{i:\'pair_early_stop\' for i in network[\'stop_indices\']}\n    pd.DataFrame([dict(id=b.IDS[i],identity=b.Y[i],role=role) for i,role in pairroles.items()]).to_csv(out/\'pair_identity_roles.csv\',index=False)\n    pd.DataFrame(network[\'history\']).to_csv(out/\'pair_training_history.csv\',index=False)\n    def bundle(role,seeds):\n        path=out/f\'{role}_bundle.joblib\'\n        if path.exists():return joblib.load(path)\n        cache.activate(roles[role]);rows=[];contexts=[];rich=[];pcs=[];pbs=[];candidates=[]\n        for seed in seeds:\n            gi,qi,truth=episode(roles[role],seed,role,(.95,111));s=p.frame(gi,qi,truth);s,r=p.through_stack(s,models)\n            a,z=v.query_features(network,b.X,b.Y,gi,s,r)\n            contexts.append(a);rich.append(z);pcs.append(r[\'pc\']);pbs.append(r[\'pb\'])\n            ids=s[\'c\'].ident.to_numpy().reshape(-1,15);candidates.append(ids)\n            for j,i in enumerate(s[\'order\']):\n                rows.append(dict(episode=seed,id=b.IDS[i],identity=b.Y[i],species=b.SP[i],truth=s[\'truth\'][j],\n                    unknown=s[\'truth\'][j]==p.NEW,candidate_present=s[\'truth\'][j] in ids[j],\n                    true_gallery_images=int((b.Y[gi]==b.Y[i]).sum()) if s[\'truth\'][j]!=p.NEW else 0))\n            print(\'Built\',role,seed,\'queries\',len(s[\'order\']),flush=True)\n        result=dict(rows=pd.DataFrame(rows),context=np.concatenate(contexts),rich=np.concatenate(rich),pc=np.concatenate(pcs),pb=np.concatenate(pbs),candidates=np.concatenate(candidates))\n        joblib.dump(result,path);return result\n    cal=bundle(\'calibration\',[args.seed+40000+j for j in range(args.calibration_episodes)])\n    # Both calibration arms have identical labels, queries and linear architecture.\n    simple=v.calibrator(cal[\'context\'],cal[\'rows\'].unknown.astype(int))\n    rich=v.calibrator(cal[\'rich\'],cal[\'rows\'].unknown.astype(int))\n    joblib.dump(dict(summary=simple,mlp=rich),out/\'calibrators.joblib\')\n    # Calibration and network fitting finish before validation labels are evaluated.\n    val=bundle(\'validation\',[args.seed+20000+j for j in range(3)])\n    d=val[\'rows\'].copy();p_simple=simple.predict_proba(val[\'context\'])[:,1];p_mlp=rich.predict_proba(val[\'rich\'])[:,1]\n    settings={\'baseline\':val[\'pb\'],\'summary_blend\':.5*(val[\'pb\']+p_simple),\n        \'mlp_blend\':.5*(val[\'pb\']+p_mlp),\'mlp_replacement\':p_mlp}\n    report={\'pair_network_parameters\':network[\'parameters\'],\'pair_fit_count\':network[\'fit_pairs\'],\'pair_stop_count\':network[\'stop_pairs\'],\n        \'primary\':\'mlp_blend\',\'methods\':{},\'episodes\':{},\'species\':{}}\n    for name,pnew in settings.items():\n        pred=v.rank(val[\'candidates\'],val[\'pc\'],pnew);d[name+\'_rr5\']=v.rr5(pred,d.truth)\n        d[name+\'_predictions\']=[\' \'.join(row) for row in pred];d[name+\'_pnew\']=pnew\n        assert all(len(set(row))==5 for row in pred)\n        report[\'methods\'][name]=metrics(d,name+\'_rr5\')\n    if args.reuse_local:\n        saved=pd.read_csv(source/\'validation_predictions.csv\').set_index([\'episode\',\'id\'])\n        assert d.baseline_predictions.tolist()==saved.loc[list(zip(d.episode,d.id)),\'baseline_predictions\'].tolist()\n        report[\'baseline_saved_prediction_parity\']=True\n    for key in (\'episodes\',\'species\'):\n        field=\'episode\' if key==\'episodes\' else \'species\'\n        report[key]={str(k):{name:metrics(z,name+\'_rr5\') for name in settings} for k,z in d.groupby(field)}\n    for name in settings:\n        report[\'methods\'][name][\'delta_map5\']=report[\'methods\'][name][\'map5\']-report[\'methods\'][\'baseline\'][\'map5\']\n        if name!=\'baseline\':report[\'methods\'][name][\'delta_identity_bootstrap95\']=bootstrap(d,name+\'_rr5\')\n    main=report[\'methods\'][\'mlp_blend\'];base=report[\'methods\'][\'baseline\']\n    report[\'promotion_gate_passed\']=bool(main[\'delta_map5\']>0 and main[\'known_map5\']>=base[\'known_map5\'] and\n        main[\'unknown_map5\']>=base[\'unknown_map5\'] and main[\'unknown_top1\']>=base[\'unknown_top1\'] and\n        all(z[\'mlp_blend\'][\'map5\']>z[\'baseline\'][\'map5\'] for z in report[\'episodes\'].values()) and\n        main[\'delta_identity_bootstrap95\'][0]>0)\n    report[\'mlp_minus_summary_map5\']=main[\'map5\']-report[\'methods\'][\'summary_blend\'][\'map5\']\n    report[\'limitations\']=[\'Existing seed2052 evaluation has been inspected before; development evidence only.\',\n        \'Repeated episodes reuse photos/identities; bootstrap clusters true identities, not retraining variance.\',\n        \'Smaller role galleries than full competition inference. Fixed settings, no validation tuning.\',\n        \'Pair probabilities are not P(known). Only separately calibrated query outputs enter final ranking.\']\n    d.to_csv(out/\'validation_predictions.csv\',index=False)\n    pd.DataFrame(splitrows).to_csv(out/\'episode_splits_this_invocation.csv\',index=False)\n    (out/\'metrics.json\').write_text(json.dumps(report,indent=2,allow_nan=False))\n    print(\'FINISHED\',json.dumps(report,indent=2),flush=True)\n\n\nif __name__==\'__main__\':\n    with threadpool_limits(limits=4,user_api=\'blas\'):main()\n', 'test_verifier.py': "import unittest\nfrom types import SimpleNamespace\nimport numpy as np\nimport pandas as pd\nimport verifier as v\n\n\nclass VerifierTests(unittest.TestCase):\n    def test_full_vector_features_are_symmetric(self):\n        x=np.array([[.1,.3,.5],[.8,.4,.2]],dtype=np.float32)\n        a=v.pair_features(x,np.array([0]),np.array([1]))\n        b=v.pair_features(x,np.array([1]),np.array([0]))\n        np.testing.assert_array_equal(a,b)\n        self.assertEqual(a.shape,(1,7))\n\n    def test_query_truth_is_not_an_inference_input(self):\n        rng=np.random.default_rng(2); x=rng.normal(size=(31,4));y=np.array([f'a{i//2}' for i in range(30)]+['query'])\n        ids=np.array([f'a{i}' for i in range(15)])\n        c=pd.DataFrame({'ident':ids,**{name:np.arange(15,dtype=float) for name in ['mxr','mxl','sx','n_flank','n_view']}})\n        s={'c':c,'q':pd.DataFrame({'sp':[1],'view':[0],'unk':[0]}),'order':np.array([30]),'truth':np.array(['a0'])}\n        network=SimpleNamespace(predict_proba=lambda z:np.tile([.4,.6],(len(z),1)))\n        scaler=SimpleNamespace(transform=lambda a:a)\n        model={'network':network,'scaler':scaler};r={'pc':np.full((1,15),1/15),'pn':np.array([.4]),'pb':np.array([.5]),'raw':np.full((1,15),.03)}\n        a=v.query_features(model,x,y,np.arange(30),s,r)\n        y[-1]='different';s['truth'][0]='new_individual';s['q']['unk']=1\n        b=v.query_features(model,x,y,np.arange(30),s,r)\n        for aa,bb in zip(a,b):np.testing.assert_array_equal(aa,bb)\n\n    def test_unknown_and_known_final_ranking(self):\n        ids=np.array([[f'a{i}' for i in range(15)]]);pc=np.array([[.8]+[.2/14]*14])\n        self.assertEqual(v.rank(ids,pc,np.array([.2]))[0,0],'a0')\n        self.assertEqual(v.rank(ids,pc,np.array([.8]))[0,0],'new_individual')\n        self.assertEqual(len(set(v.rank(ids,pc,np.array([.2]))[0])),5)\n\n\nif __name__=='__main__':unittest.main()\n", 'requirements.txt': 'numpy>=1.26,<2.5\npandas>=2.1,<3\nscipy>=1.11,<2\nscikit-learn>=1.4,<2\nlightgbm>=4,<5\nopencv-python>=4.8,<5\njoblib>=1.3,<2\nthreadpoolctl>=3,<4\n', 'persistence.py': '"""Portable, checksummed Model 11 checkpoint bundles; never includes input data."""\nfrom pathlib import Path,PurePosixPath\nimport json,hashlib,zipfile,os,platform,tempfile,shutil\nREQUIRED=(\'config.json\',\'baseline_models.joblib\',\'verifier.joblib\',\'calibrators.joblib\')\n\ndef digest(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda:stream.read(1024*1024),b\'\'):h.update(block)\n    return h.hexdigest()\n\ndef save_bundle(run,code,archive):\n    run,code,archive=map(Path,(run,code,archive));archive.parent.mkdir(parents=True,exist_ok=True)\n    if (run/\'config.json\').exists():\n        config=json.loads((run/\'config.json\').read_text())\n        for name in (\'dalvin_core.py\',\'data_protocol.py\',\'verifier.py\'):\n            expected=config.get(\'sources\',{}).get(name)\n            if expected and (not (code/name).exists() or digest(code/name)!=expected):\n                raise ValueError(f\'Backup source does not match the trained checkpoint: {name}\')\n    files=[]\n    for label,folder in [(\'run\',run),(\'code\',code)]:\n        if folder.exists():\n            for path in sorted(folder.iterdir()):\n                if path.is_file() and path.suffix not in (\'.tmp\',\'.zip\',\'.pyc\'):\n                    files.append((f\'{label}/{path.name}\',path))\n    if not files:raise FileNotFoundError(\'No checkpoints or source files to save\')\n    manifest=dict(format=1,python=platform.python_version(),\n        complete_for_submission=all((run/name).is_file() for name in REQUIRED),\n        missing_for_submission=[name for name in REQUIRED if not (run/name).is_file()],files={})\n    temp=archive.with_suffix(\'.tmp\')\n    try:\n        with zipfile.ZipFile(temp,\'w\',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as z:\n            for name,path in files:\n                before=digest(path);z.write(path,name)\n                if digest(path)!=before:raise RuntimeError(f\'File changed during backup: {path}; retry after computation stops\')\n                manifest[\'files\'][name]=before\n            z.writestr(\'manifest.json\',json.dumps(manifest,indent=2))\n        # Verify the actual archive contents before publishing the backup.\n        with zipfile.ZipFile(temp) as z:\n            for name,expected in manifest[\'files\'].items():\n                if hashlib.sha256(z.read(name)).hexdigest()!=expected:raise ValueError(\'Backup checksum mismatch\')\n        os.replace(temp,archive)\n    finally:\n        if temp.exists():temp.unlink()\n    print(\'Saved checkpoint bundle:\',archive,\'complete for submission:\',manifest[\'complete_for_submission\'])\n    return manifest\n\ndef restore_bundle(archive,run,code):\n    run,code=Path(run),Path(code)\n    with zipfile.ZipFile(archive) as z:\n        manifest=json.loads(z.read(\'manifest.json\'))\n        if manifest.get(\'format\')!=1:raise ValueError(\'Unknown checkpoint bundle format\')\n        if len(z.namelist())!=len(set(z.namelist())):raise ValueError(\'Duplicate archive members\')\n        if set(z.namelist())!=set(manifest[\'files\'])|{\'manifest.json\'}:raise ValueError(\'Unexpected archive members\')\n        for name,expected in manifest[\'files\'].items():\n            parts=PurePosixPath(name).parts\n            if len(parts)!=2 or parts[0] not in (\'run\',\'code\') or parts[1] in (\'.\',\'..\') or \'\\\\\' in name or \':\' in name:\n                raise ValueError(\'Unsafe archive path\')\n            if hashlib.sha256(z.read(name)).hexdigest()!=expected:raise ValueError(f\'Corrupt bundle: {name}\')\n            destination=(run if parts[0]==\'run\' else code)/parts[1]\n            if destination.exists() and digest(destination)!=expected:\n                raise ValueError(f\'Restore would overwrite different files at {destination}. Use empty OUTPUT/CODE folders.\')\n        # All members and existing destinations are verified before writing.\n        for name in manifest[\'files\']:\n            group,filename=PurePosixPath(name).parts\n            destination=(run if group==\'run\' else code)/filename\n            destination.parent.mkdir(parents=True,exist_ok=True)\n            destination.write_bytes(z.read(name))\n    print(\'Restored verified checkpoints and sources:\',run)\n    return manifest\n'}
EXPECTED_HASHES = {'dalvin_core.py': 'bfeb3384aefce320d2980e68f85b00ea5f109b2c2d4cee7e70d79f05e17a3f1f', 'data_protocol.py': 'fba94b7e73b87e60ee83f3bcb4a5cc01b48082533d3c7d6f420e4e305da4ed72', 'verifier.py': '2c0ad83e6ffeecb4b8e62145fd0626c973efeaf794fe6125765eda1f99d3e5d5', 'run.py': '034fd860e6fb2c51ca3b18e3bf159fafb78644faf012c070ca5bdc6d574f5997', 'test_verifier.py': '81148d59245a8240af40a7e406d51d51dc9b4367b77f8a2d7cf2733f9b3be815', 'requirements.txt': '1f3e15fb9d643d26b3df18454c1e238704e87298c29eccccd2f8bc285ff65d30', 'persistence.py': '58df1214638d125fe135cb31250344cb0013b730abb983ae8b912fb5a8689088'}
for name, text in SOURCES.items():
    assert hashlib.sha256(text.encode()).hexdigest() == EXPECTED_HASHES[name]
    (CODE/name).write_text(text, encoding='utf-8')
print('Verified and extracted', len(SOURCES), 'embedded source files')


In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location('model11_persistence', CODE/'persistence.py')
persistence = importlib.util.module_from_spec(spec)
spec.loader.exec_module(persistence)
if MODE not in ('train', 'submission'):
    raise ValueError("MODE must be 'train' or 'submission'")
if RESTORE_ARCHIVE is None and MODE == 'submission':
    matches = list(Path('/kaggle/input').rglob('model11_checkpoint_bundle.zip'))
    if len(matches) == 1: RESTORE_ARCHIVE = matches[0]
    elif len(matches) > 1: raise ValueError(f'Set RESTORE_ARCHIVE explicitly: {matches}')
if RESTORE_ARCHIVE is not None:
    # Restore exact training sources into a separate folder, protecting current code.
    CODE = Path('/kaggle/working/model11_restored_code')
    restored = persistence.restore_bundle(RESTORE_ARCHIVE, OUTPUT, CODE)
    if MODE == 'submission' and not restored['complete_for_submission']:
        raise RuntimeError(f"This backup is partial: {restored['missing_for_submission']}")
if MODE == 'submission':
    missing = [name for name in persistence.REQUIRED if not (OUTPUT/name).exists()]
    if missing: raise RuntimeError(f'Attach a saved checkpoint bundle and set RESTORE_ARCHIVE. Missing: {missing}')
    print('Submission mode: training and validation will be skipped.')


In [ ]:
if INSTALL_REQUIREMENTS:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(CODE/'requirements.txt')], check=True)
import importlib
for name in ['numpy','pandas','scipy','sklearn','lightgbm','cv2','joblib','threadpoolctl']:
    module = importlib.import_module(name)
    print(name, getattr(module, '__version__', 'import OK'))
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(CODE), '-p', 'test_verifier.py', '-v'], check=True)


In [ ]:
root = Path('/kaggle/input')
if DATA_ROOT is None:
    options = [p.parent for p in root.rglob('miewid_features.npy')
               if all((p.parent/name).exists() for name in ['train.csv','test.csv','miewid_ids.csv'])]
    if len(options) != 1:
        raise RuntimeError(f'Set DATA_ROOT explicitly; found metadata folders: {options}')
    DATA_ROOT = options[0]
DATA_ROOT = Path(DATA_ROOT)
# Make a small input view so SIFT assets can be mounted in a separate Kaggle dataset.
INPUT = Path('/kaggle/working/model11_input')
INPUT.mkdir(exist_ok=True)
for name in ['train.csv','test.csv','miewid_ids.csv','miewid_features.npy']:
    target = DATA_ROOT/name
    if not target.exists(): raise FileNotFoundError(target)
    link = INPUT/name
    if link.exists() and link.resolve() != target.resolve(): raise RuntimeError(f'Stale input link: {link}')
    if not link.exists(): link.symlink_to(target)
for split in ['train','test']:
    name = f'sift_features_{split}.npz'
    nearby = [p/name for p in [DATA_ROOT, DATA_ROOT.parent, DATA_ROOT.parent.parent] if (p/name).exists()]
    options = nearby[:1] or list(root.rglob(name))
    if len(options) != 1:
        raise RuntimeError(f'Attach the full competition SIFT input; ambiguous/missing {name}: {options}')
    target = options[0]; link = INPUT/name
    if link.exists() and link.resolve() != target.resolve(): raise RuntimeError(f'Stale SIFT link: {link}')
    if not link.exists(): link.symlink_to(target, target_is_directory=target.is_dir())
    print(split, target)
print('Metadata:', DATA_ROOT)


Run the next cell to completion. It saves the identity assignments, training history, source hashes, baseline and verifier checkpoints, calibrated models and per-query validation predictions. Changing settings/source requires a new OUTPUT folder. No competition submission is generated or uploaded.

In [ ]:
command = [sys.executable, '-u', str(CODE/'run.py'),
    '--data', str(INPUT), '--output', str(OUTPUT), '--seed', str(SEED),
    '--workers', str(WORKERS), '--epochs', str(MLP_MAX_EPOCHS),
    '--max-pairs', str(MAX_TRAIN_PAIRS), '--baseline-sims', str(BASELINE_EPISODES_PER_STAGE),
    '--calibration-episodes', str(CALIBRATION_EPISODES), '--trees', str(CANDIDATE_TREES)]
from IPython.display import FileLink, display
if MODE == 'train':
    try:
        subprocess.run(command, check=True)
    finally:
        if (OUTPUT/'config.json').exists():
            persistence.save_bundle(OUTPUT, CODE, BACKUP)
            display(FileLink(str(BACKUP)))
else:
    print('Using restored models; skipped training/validation.')


In [ ]:
if MODE == "train":
    import pandas as pd
    report = json.loads((OUTPUT/'metrics.json').read_text())
    display(pd.DataFrame(report['methods']).T)
    print('MLP gain beyond summary-only calibration:', report['mlp_minus_summary_map5'])
    print('Strict primary promotion gate passed:', report['promotion_gate_passed'])
    for episode, scores in report['episodes'].items():
        print('Episode', episode)
        display(pd.DataFrame(scores).T)
    for species, scores in report['species'].items():
        print('Species', species)
        display(pd.DataFrame(scores).T)


In [ ]:
if MODE == "train":
    # Independent score and identity-role checks.
    pred = pd.read_csv(OUTPUT/'validation_predictions.csv')
    roles = pd.read_csv(OUTPUT/'identity_roles.csv')
    assert roles.groupby('identity').role.nunique().max() == 1
    pair_roles = pd.read_csv(OUTPUT/'pair_identity_roles.csv')
    assert pair_roles.groupby('identity').role.nunique().max() == 1
    for method, metrics in report['methods'].items():
        scores = []
        for row in pred.itertuples(index=False):
            labels = getattr(row, method+'_predictions').split()
            assert len(labels) == len(set(labels)) == 5
            score = next((1/(i+1) for i, label in enumerate(labels) if label == row.truth), 0.)
            assert abs(score-getattr(row, method+'_rr5')) < 1e-12
            scores.append(score)
        assert abs(sum(scores)/len(scores)-metrics['map5']) < 1e-12
    print('All per-query scores and pooled MAP@5 values recompute correctly.')
    from IPython.display import FileLink
    display(FileLink(str(OUTPUT/'metrics.json')))
    display(FileLink(str(OUTPUT/'validation_predictions.csv')))


## Keep the trained models across sessions

**Files in `/kaggle/working` alone are temporary. Download `model11_checkpoint_bundle.zip` using the link above, or preserve it in the saved notebook outputs before ending the session.** The archive includes baseline models, MLP weights/scaler, calibrators, the current SIFT cache, configuration, metrics and source snapshots. Every file has a verified SHA-256 checksum.

For a later submission: attach this ZIP as Kaggle input alongside the competition data, set `MODE = 'submission'` in the first settings cell, and Run All. The archive is discovered automatically if there is exactly one matching ZIP. Otherwise set `RESTORE_ARCHIVE` explicitly. Training and validation are skipped. The final submission cell uses the restored checkpoints and reuses cached SIFT pairs. Pairs needed only for full-gallery inference still need computing once.

If training failed, the backup may be partial; its manifest lists missing files. A partial backup can preserve completed stages/cache for resuming identical training, but cannot generate submissions until all required checkpoints exist. Save/download the refreshed bundle after submission too; it includes the expanded full-gallery SIFT cache.

Interpretation: compare `mlp_blend` with `baseline` for the primary result, and with `summary_blend` to assess the incremental value of vector-pair evidence. Promotion requires a positive overall gain with a positive identity-cluster bootstrap lower bound, positive gain in every episode, and no decrease in known MAP@5, actual-new MAP@5 or actual-new top-one detection. A failed gate means retain the baseline. Passing is development evidence and still does not guarantee a competition gain. Report negative results too; do not select a different blend from validation output.

## Optional: generate competition submissions
Run the next cell only when you want test predictions. It uses the saved models and the full labelled gallery. Full-population SIFT verification can take substantial time; its cache is saved under the existing output folder. This creates the requested MLP-blend CSV and a baseline control. It does not upload to Kaggle.


In [ ]:
if globals().get("MODE") == "submission" or globals().get("GENERATE_SUBMISSION", False):
    # Paste this ENTIRE cell at the end of the completed Model 11 Kaggle notebook.
    from pathlib import Path
    import subprocess, sys
    from IPython.display import FileLink, display
    
    # Optional: if several saved runs are attached, set OUTPUT to the desired folder.
    # Otherwise checkpoints are discovered automatically in working files and saved inputs.
    CHECKPOINT_CODE = globals().get('CODE')
    
    RECOVERY_SOURCE = '"""Recover completed Model 11 outputs and Kaggle inputs without refitting."""\nfrom pathlib import Path\nimport json,hashlib,shutil,zipfile\n\nREQUIRED=(\'config.json\',\'baseline_models.joblib\',\'verifier.joblib\',\'calibrators.joblib\')\n\n\ndef recover_archives(roots,working):\n    """Recognize checkpoint ZIPs by content, including renamed uploaded files."""\n    archives=[]\n    for root in roots:\n        if root.exists():\n            archives.extend(path for path in root.rglob(\'*\') if path.is_file() and path.suffix.lower()==\'.zip\')\n    recovered=[]\n    for archive in archives:\n        try:\n            with zipfile.ZipFile(archive) as z:\n                if \'manifest.json\' not in z.namelist():continue\n                manifest=json.loads(z.read(\'manifest.json\'))\n                if manifest.get(\'format\')!=1 or not all(\'run/\'+name in manifest.get(\'files\',{}) for name in REQUIRED):continue\n                # Verify all payloads before exposing any checkpoint files.\n                for name,expected in manifest[\'files\'].items():\n                    parts=name.split(\'/\')\n                    if len(parts)!=2 or parts[0] not in (\'run\',\'code\') or parts[1] in (\'\',\'..\',\'.\') or \'\\\\\' in name or \':\' in name:\n                        raise ValueError(\'Unsafe checkpoint archive member\')\n                    if hashlib.sha256(z.read(name)).hexdigest()!=expected:raise ValueError(f\'Checkpoint checksum failed: {name}\')\n                key=hashlib.sha256(z.read(\'manifest.json\')).hexdigest()[:12]\n                run=working/f\'model11_restored_{key}\'\n                for name,expected in manifest[\'files\'].items():\n                    group,filename=name.split(\'/\')\n                    target=run/filename if group==\'run\' else run/\'saved_sources\'/filename\n                    if target.exists() and hashlib.sha256(target.read_bytes()).hexdigest()!=expected:\n                        raise ValueError(f\'Restore would overwrite another file: {target}\')\n                for name in manifest[\'files\']:\n                    group,filename=name.split(\'/\')\n                    target=run/filename if group==\'run\' else run/\'saved_sources\'/filename\n                    target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(z.read(name))\n                print(\'Restored checkpoint ZIP:\',archive,\'->\',run)\n                recovered.append(run)\n        except zipfile.BadZipFile:continue\n    return sorted(set(recovered))\n\n\ndef find_runs(roots):\n    complete=[];partial=[]\n    for root in roots:\n        if not root.exists():continue\n        for file in root.rglob(\'config.json\'):\n            try:cfg=json.loads(file.read_text())\n            except (OSError,ValueError):continue\n            if cfg.get(\'primary\')!=\'mlp_blend\':continue\n            missing=[name for name in REQUIRED if not (file.parent/name).is_file()]\n            if missing:partial.append((str(file.parent),missing))\n            else:complete.append(file.parent)\n    return sorted(set(complete)),partial\n\n\ndef prepare(sources,preferred=None,roots=None,working=Path(\'/kaggle/working\')):\n    roots=roots or [working,Path(\'/kaggle/input\')]\n    runs,partial=find_runs(roots)\n    if not runs:\n        recover_archives(roots,working)\n        runs,partial=find_runs(roots)\n    preferred=Path(preferred) if preferred is not None else None\n    if preferred in runs:selected=preferred\n    elif len(runs)==1:selected=runs[0]\n    elif not runs:\n        raise FileNotFoundError(\'No complete Model 11 checkpoints found in /kaggle/working or /kaggle/input. \'\n            \'Attach the saved OUTPUT of your completed training notebook via Kaggle Add Input, then rerun this cell. \'\n            \'If those checkpoint files were never saved, the metric table cannot recreate them and training must be rerun. \'\n            f\'Partial runs and missing files: {partial}. \'\n            f\'Attached input folders: {[str(path) for root in roots if root.exists() for path in root.iterdir() if path.is_dir()]}\')\n    else:raise ValueError(f\'Multiple completed runs found. Set OUTPUT to the intended folder and rerun: {runs}\')\n    print(\'Found completed checkpoints:\',selected)\n    output=selected\n    if not selected.is_relative_to(working):\n        output=working/\'model11_submission_from_saved\';output.mkdir(parents=True,exist_ok=True)\n        for name in list(REQUIRED)+[f.name for f in selected.glob(\'sift_mutual_*.npz\')]:\n            shutil.copy2(selected/name,output/name)\n        print(\'Copied checkpoints into writable folder:\',output)\n    config=json.loads((output/\'config.json\').read_text())\n    code=working/\'model11_submission_code\';code.mkdir(parents=True,exist_ok=True)\n    for name,text in sources.items():\n        expected=config[\'sources\'][name]\n        variants=[text.encode(),text.replace(\'\\n\',\'\\r\\n\').encode()]\n        chosen=next((raw for raw in variants if hashlib.sha256(raw).hexdigest()==expected),None)\n        if chosen is None:raise ValueError(f\'Training source differs for {name}; use the sources from that saved notebook version\')\n        (code/name).write_bytes(chosen)\n    input_root=Path(\'/kaggle/input\')\n    options=[file.parent for file in input_root.rglob(\'miewid_features.npy\')\n             if all((file.parent/name).exists() for name in (\'train.csv\',\'test.csv\',\'miewid_ids.csv\'))]\n    if len(options)!=1:raise ValueError(f\'Attach the competition feature data (one metadata folder expected): {options}\')\n    data=options[0];inputs=working/\'model11_submission_input\';inputs.mkdir(exist_ok=True)\n    for name in (\'train.csv\',\'test.csv\',\'miewid_ids.csv\',\'miewid_features.npy\'):\n        dest=inputs/name;target=data/name\n        if dest.is_symlink() and not dest.exists():dest.unlink()\n        if not dest.exists():dest.symlink_to(target)\n        if dest.resolve()!=target.resolve():raise ValueError(f\'Stale input link: {dest}\')\n    for split in (\'train\',\'test\'):\n        name=f\'sift_features_{split}.npz\'\n        nearby=[base/name for base in (data,data.parent,data.parent.parent) if (base/name).exists()]\n        options=nearby[:1] or list(input_root.rglob(name))\n        if len(options)!=1:raise ValueError(f\'Attach full competition SIFT input: {name}: {options}\')\n        target=options[0];dest=inputs/name\n        if dest.is_symlink() and not dest.exists():dest.unlink()\n        if not dest.exists():dest.symlink_to(target,target_is_directory=target.is_dir())\n        if dest.resolve()!=target.resolve():raise ValueError(f\'Stale SIFT link: {dest}\')\n    import pandas as pd\n    ids=set(pd.read_csv(inputs/\'test.csv\').id)\n    candidates=[data/\'sample_submission.csv\'] if (data/\'sample_submission.csv\').exists() else list(input_root.rglob(\'sample_submission.csv\'))\n    matches=[]\n    for file in candidates:\n        frame=pd.read_csv(file)\n        if \'id\' in frame and frame.id.is_unique and set(frame.id)==ids:matches.append(file)\n    if len(matches)!=1:raise ValueError(f\'Cannot locate an unambiguous sample submission: {matches}\')\n    return code,output,inputs,matches[0]\n'
    TRAINING_SOURCES = {'dalvin_core.py': '"""Dalvin Model 3d functions; raw evidence can use a frozen nuisance projection."""\nimport os\nimport re\nimport cv2\nimport lightgbm as lgb\nimport numpy as np\nimport pandas as pd\nfrom multiprocessing import Pool\nfrom scipy import sparse\nfrom scipy.sparse.csgraph import connected_components\nfrom sklearn.decomposition import PCA\nfrom sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA\nK=15\nNEIGHBOURS=150\nMATCH_RATIO=.85\nRANSAC_PX=15.\nSIFT_PULL=.05\nVIEWS=[\'left\',\'right\',\'front\',\'back\',\'frontleft\',\'frontright\',\'backleft\',\'backright\',\'up\',\'down\']\nFLANK=np.array([0,1,2,2,0,1,0,1,2,2])\nREGIMES=[None,(.95,111),(.99,89),(.9,67)]\nPARAMS=dict(objective=\'binary\',learning_rate=.05,num_leaves=31,min_child_samples=30,subsample=.8,subsample_freq=1,colsample_bytree=.8,n_estimators=400,verbose=-1,random_state=42,n_jobs=4)\nINDEX_COLS=re.compile(r\'^(nb|nw|sb|sw)\\d$\')\n\n\n\n\n\n\n\ndef unit(a):\n    return a / np.linalg.norm(a, axis=1, keepdims=True)\n\n\ndef embed(gallery, labels, queries):\n    pca = PCA(min(384, gallery.shape[1], len(gallery) + len(queries) - 1), random_state=0).fit(np.r_[gallery, queries])      # unsupervised, sees the queries too\n    lda = LDA(solver="eigen", shrinkage=0.3).fit(pca.transform(gallery), labels)\n    project = lambda a: unit(lda.transform(pca.transform(a)))\n    return project(gallery), project(queries)\n\n\ndef per_identity(sim, inv, n):\n    """Per query: max and mean-of-top-2 similarity to each identity\'s gallery images."""\n    best, top2 = np.empty((sim.shape[0], n)), np.empty((sim.shape[0], n))\n    for j in range(n):\n        s = sim[:, inv == j]\n        best[:, j], top2[:, j] = s.max(1), np.sort(s, 1)[:, -2:].mean(1)\n    return best, top2\n\n\ndef species_features(gi, qi, sp):\n    """Query frame and (query, candidate) frame for one species; gi/qi are global image indices."""\n    Rg, Rq, yg, vg, vq = X[gi], X[qi], Y[gi], VIEW[gi], VIEW[qi]\n    Lg, Lq = embed(Rg, yg, Rq)\n    Rg, Rq = NX[gi], NX[qi]  # Frozen nuisance representation for raw evidence only.\n    ids, inv = np.unique(yg, return_inverse=True)\n    n, nq = len(ids), len(qi)\n    sim_l, sim_r = Lq @ Lg.T, Rq @ Rg.T\n    lda1, lda2 = per_identity(sim_l, inv, n)\n    raw1, raw2 = per_identity(sim_r, inv, n)\n    centroid = unit(np.stack([Lg[inv == j].mean(0) for j in range(n)]))\n    cen = Lq @ centroid.T\n    inl_g, good_g = INL[qi][:, gi].toarray().astype(np.float32), GOOD[qi][:, gi].toarray().astype(np.float32)\n    sift1, sift2 = per_identity(inl_g, inv, n)\n    good1, _ = per_identity(good_g, inv, n)\n    support = (inl_g >= 7).astype(np.float32) @ np.eye(n, dtype=np.float32)[inv]     # gallery images confirmed per identity\n    same_view = np.full((nq, n), np.nan)\n    for j in range(n):\n        for v in np.unique(vq):\n            cols = np.where((inv == j) & (vg == v))[0]\n            if len(cols):\n                same_view[vq == v, j] = sim_l[np.ix_(vq == v, cols)].max(1)\n    # viewpoint coverage: flanks of one animal look different, so unseen sides cannot be matched\n    by_view = np.stack([np.bincount(inv[vg == v], minlength=n) for v in range(len(VIEWS))], 1)\n    by_flank = np.stack([by_view[:, FLANK == f].sum(1) for f in range(3)], 1)\n    seen = vq >= 0\n    n_view = np.where(seen[:, None], by_view[:, vq.clip(0)].T, np.nan)\n    n_flank = np.where(seen[:, None], by_flank[:, FLANK[vq.clip(0)]].T, np.nan)\n    # transductive: neighbours among the queries themselves\n    qq_r, qq_l, qq_s = Rq @ Rq.T, Lq @ Lq.T, INL[qi][:, qi].toarray().astype(np.float32)\n    np.fill_diagonal(qq_r, -1), np.fill_diagonal(qq_l, -1)\n    nb, sb = np.argsort(-qq_l, 1)[:, :5], np.argsort(-qq_s, 1)[:, :5]\n    nbs, sbs = np.take_along_axis(qq_l, nb, 1), np.take_along_axis(qq_s, sb, 1)\n    t_l = np.take_along_axis(lda1, np.argsort(-lda1, 1)[:, :3], 1)\n    t_r = -np.sort(-raw1, 1)[:, :3]\n    t_s = -np.sort(-sift1, 1)[:, :3]\n    g_top = -np.sort(-sim_l, 1)\n    q = pd.DataFrame({\n        "sp": int(sp == "hyena"), "view": vq, "ng": len(gi),\n        "l1": t_l[:, 0], "l2": t_l[:, 1], "l3": t_l[:, 2], "r1": t_r[:, 0], "r2": t_r[:, 1], "r3": t_r[:, 2],\n        "lm12": t_l[:, 0] - t_l[:, 1], "rm12": t_r[:, 0] - t_r[:, 1],\n        "gd5": g_top[:, :5].mean(1), "gd20": g_top[:, :20].mean(1),\n        "qq1": nbs[:, 0], "qq3": nbs[:, :3].mean(1), "qq5": nbs.mean(1), "qqr1": qq_r.max(1),\n        "qn6": (qq_r > .6).sum(1), "qn7": (qq_r > .7).sum(1), "qn8": (qq_r > .8).sum(1),\n        "qq_minus_g": nbs[:, 0] - t_l[:, 0], "nb_g1": t_l[nb[:, 0], 0],\n        "s1": t_s[:, 0], "s2": t_s[:, 1], "s3": t_s[:, 2], "sg": good_g.max(1),\n        "qs1": sbs[:, 0], "qs3": sbs[:, :3].mean(1), "qsn10": (qq_s >= 10).sum(1), "qsn20": (qq_s >= 20).sum(1),\n        "qs_minus_s": sbs[:, 0] - t_s[:, 0], "nbs_s1": t_s[sb[:, 0], 0],\n    })\n    for k in range(5):\n        q[f"nb{k}"], q[f"nw{k}"], q[f"sb{k}"], q[f"sw{k}"] = nb[:, k], nbs[:, k], sb[:, k], sbs[:, k]\n    # candidates: best identities by embedding similarity, lightly boosted by SIFT evidence\n    cand = np.argsort(-(lda1 + SIFT_PULL * np.tanh(sift1 / 15)), 1)[:, :K]\n    r, c = np.repeat(np.arange(nq), K), cand.ravel()\n    nb_ev = np.max(np.minimum(nbs[:, :, None], np.stack([lda1[nb[:, k]] for k in range(5)], 1)), 1)\n    sb_ev = np.max(np.minimum(sbs[:, :, None], np.stack([sift1[sb[:, k]] for k in range(5)], 1)), 1)\n    rank_of = lambda m: (-m).argsort(1).argsort(1)[r, c]\n    cd = pd.DataFrame({\n        "q": r, "c": c, "rank": np.tile(np.arange(K), nq), "ident": ids[c],\n        "mxr": raw1[r, c], "m2r": raw2[r, c], "mxl": lda1[r, c], "m2l": lda2[r, c], "cl": cen[r, c],\n        "gapl": lda1[r, c] - t_l[r, 0], "gapr": raw1[r, c] - t_r[r, 0], "same": same_view[r, c],\n        "cnt": np.bincount(inv)[c], "nbev": nb_ev[r, c], "sbev": sb_ev[r, c],\n        "rk_r": rank_of(raw1), "rk_c": rank_of(cen), "rk_s": rank_of(sift1),\n        "n_view": n_view[r, c], "n_flank": n_flank[r, c], "support": support[r, c], "sx": sift1[r, c], "s2x": sift2[r, c], "gx": good1[r, c], "sgap": sift1[r, c] - t_s[r, 0],\n    })\n    return q, cd\n\n\ndef build_frames(gi, qi):\n    qs, cs, order, off = [], [], [], 0\n    for sp in ("hyena", "leopard"):\n        g, qq = gi[SP[gi] == sp], qi[SP[qi] == sp]\n        q, c = species_features(g, qq, sp)\n        c["q"] += off\n        q[[f"{p}{k}" for p in ("nb", "sb") for k in range(5)]] += off\n        qs.append(q), cs.append(c), order.append(qq)\n        off += len(qq)\n    return dict(q=pd.concat(qs, ignore_index=True), c=pd.concat(cs, ignore_index=True), order=np.concatenate(order))\n\n\ndef duplicate_groups(ix, link):\n    """Group images of one identity that are near-duplicates (same encounter); each image alone if link is None."""\n    if link is None:\n        return np.arange(len(ix))\n    edges = (X[ix] @ X[ix].T > link[0]) | (INL[ix][:, ix].toarray() >= link[1])\n    return connected_components(sparse.csr_matrix(edges))[1]\n\n\ndef columns(s):\n    qf = [c for c in s["q"].columns if c != "unk" and not INDEX_COLS.match(c)]\n    cf = [c for c in s["c"].columns if c not in ("q", "c", "ident", "y")]\n    return qf, cf\n\n\ndef candidate_matrix(s, qf, cf):\n    return pd.concat([s["c"][cf].reset_index(drop=True), s["q"].iloc[s["c"].q.to_numpy()][qf].reset_index(drop=True)], axis=1)\n\n\ndef fit(sims):\n    """new-individual detector, closed-set candidate scorer, and all-query candidate scorer."""\n    qf, cf = columns(sims[0])\n    known = lambda s: s["q"].unk.to_numpy()[s["c"].q.to_numpy()] == 0\n    cx = pd.concat([candidate_matrix(s, qf, cf) for s in sims])\n    cy = np.concatenate([s["c"].y.to_numpy() for s in sims])\n    ck = np.concatenate([known(s) for s in sims])\n    detector = lgb.LGBMClassifier(**{**PARAMS, "n_estimators": 300, "num_leaves": 15}).fit(\n        pd.concat([s["q"][qf] for s in sims]), np.concatenate([s["q"].unk.to_numpy() for s in sims]))\n    return detector, lgb.LGBMClassifier(**PARAMS).fit(cx[ck], cy[ck]), lgb.LGBMClassifier(**PARAMS).fit(cx, cy), (qf, cf)\n\n\ndef predict(models, s):\n    detector, closed, anyq, (qf, cf) = models\n    nq, cx = len(s["q"]), candidate_matrix(s, qf, cf)\n    p_new = detector.predict_proba(s["q"][qf])[:, 1]\n    raw = anyq.predict_proba(cx)[:, 1].reshape(nq, K)\n    within = closed.predict_proba(cx)[:, 1].reshape(nq, K)\n    within /= np.maximum(within.sum(1, keepdims=True), 1e-9)\n    return dict(pn=p_new, raw=raw, pc=within, pb=(p_new + np.clip(1 - raw.sum(1), 0, 1)) / 2)\n\n\ndef rank(s, r):\n    """Top-5 labels: P(new) vs (1 - P(new)) * P(identity | known)."""\n    nq = len(s["q"])\n    labels = np.concatenate([s["c"].ident.to_numpy(dtype=object).reshape(nq, K), np.full((nq, 1), "new_individual", dtype=object)], 1)\n    scores = np.concatenate([(1 - r["pb"])[:, None] * r["pc"], r["pb"][:, None]], 1)\n    return np.take_along_axis(labels, np.argsort(-scores, 1)[:, :5], 1)\n\n\ndef map5(preds, truth):\n    return np.mean([next((1 / (i + 1) for i, p in enumerate(row) if p == t), 0.0) for row, t in zip(preds, truth)])\n\n\ndef augment(s, r):\n    """Add the previous round\'s predictions for each query and for its nearest query neighbours."""\n    q, c, nq = s["q"].copy(), s["c"].copy(), len(s["q"])\n    codes = pd.factorize(c.ident)[0].reshape(nq, K)\n    q["r_pn"], q["r_pb"], q["r_rawsum"], q["r_rawmax"] = r["pn"], r["pb"], r["raw"].sum(1), r["raw"].max(1)\n    c["r_raw"], c["r_pc"] = r["raw"].ravel(), r["pc"].ravel()\n    for kind, weight, scale in (("nb", "nw", 1.0), ("sb", "sw", 30.0)):     # embedding / SIFT neighbours\n        evidence, support, pb, w, top = np.zeros((nq, K)), np.zeros((nq, K)), [], [], []\n        for k in range(5):\n            idx, sim = q[f"{kind}{k}"].to_numpy(), q[f"{weight}{k}"].to_numpy() / scale\n            wk = np.clip(sim - 0.4, 0, None) if kind == "nb" else np.clip(sim, 0, 1)\n            agree = ((codes[idx][:, None, :] == codes[:, :, None]) * r["raw"][idx][:, None, :]).sum(2)\n            evidence = np.maximum(evidence, np.minimum(agree, np.clip(sim, 0, 1)[:, None]))\n            support += wk[:, None] * agree\n            pb.append(r["pb"][idx]), w.append(wk), top.append(r["raw"][idx].max(1))\n        pb, w = np.stack(pb, 1), np.stack(w, 1)\n        q[f"{kind}_pb0"], q[f"{kind}_pbw"] = pb[:, 0], (pb * w).sum(1) / np.maximum(w.sum(1), 1e-6)\n        q[f"{kind}_pbmin"], q[f"{kind}_w"], q[f"{kind}_rm0"] = np.where(w > 0, pb, 1).min(1), w.sum(1), top[0]\n        c[f"r_{kind}ev"], c[f"r_{kind}sum"] = evidence.ravel(), support.ravel()\n    return dict(s, q=q, c=c)\n', 'data_protocol.py': '"""Data loading, role-local mutual SIFT and identity-disjoint episode helpers."""\nimport os\nimport hashlib\nfrom pathlib import Path\nfrom concurrent.futures import ThreadPoolExecutor\nfrom functools import lru_cache\nfrom threading import Lock\nimport numpy as np\nimport pandas as pd\nimport cv2\nfrom scipy import sparse\nimport dalvin_core as b\nNEW=\'new_individual\'\ndef find_sift(root, split):\n    """Kaggle mounts SIFT beside the metadata folder, sometimes as extracted NPZ directories."""\n    name = f"sift_features_{split}.npz"\n    for parent in (root, root.parent, root.parent.parent):\n        candidate = parent / name\n        if candidate.exists():\n            return candidate\n    raise FileNotFoundError(f"Cannot find {name} in {root} or its two parent directories. "\n                            "Attach the full projectwild3 competition input, including both SIFT folders.")\n\n\nclass SiftSource:\n    def __init__(self, path):\n        self.archive = None\n        self.files = {}\n        if path.is_file():\n            self.archive = np.load(path)\n        else:\n            for file in path.rglob("*.npy"):\n                if file.stem in self.files:\n                    raise ValueError(f"Duplicate SIFT key {file.stem} in {path}")\n                self.files[file.stem] = file\n            if not self.files:\n                raise ValueError(f"No extracted .npy SIFT arrays in {path}")\n\n    def __getitem__(self, key):\n        return self.archive[key] if self.archive is not None else np.load(self.files[key], allow_pickle=False)\n\n\ndef load_data(root):\n    train = pd.read_csv(root / "train.csv")\n    test = pd.read_csv(root / "test.csv")\n    ids = pd.read_csv(root / "miewid_ids.csv")\n    all_df = pd.concat([train, test.assign(identity=None)], ignore_index=True)\n    if not all_df.id.is_unique or not ids.id.is_unique:\n        raise ValueError("Duplicate image IDs")\n    for col in ("species", "viewpoint", "identity"):\n        if col not in all_df:\n            raise ValueError(f"Missing column: {col}")\n    if train.identity.isna().any() or not set(all_df.species) <= {"hyena", "leopard"}:\n        raise ValueError("Invalid identity/species metadata")\n    raw = np.load(root / "miewid_features.npy", mmap_mode="r")\n    if len(raw) != len(ids) or set(ids.id) != set(all_df.id):\n        raise ValueError("Feature ID mapping does not match metadata")\n    ix = pd.Index(ids.id).get_indexer(all_df.id)\n    X = np.asarray(raw[ix], dtype=np.float32)\n    if not np.isfinite(X).all() or np.any(np.linalg.norm(X, axis=1) == 0):\n        raise ValueError("Features contain nonfinite values or zero vectors")\n    b.X = b.unit(X)\n    b.IDS = all_df.id.to_numpy()\n    b.Y = all_df.identity.to_numpy(dtype=object)\n    b.SP = all_df.species.to_numpy()\n    b.VIEW = all_df.viewpoint.map({v: i for i, v in enumerate(b.VIEWS)}).fillna(-1).to_numpy(int)\n    return train, test, all_df\n\n\nclass SiftCache:\n    """Role-local neighbour selection; thread workers share lazily loaded descriptors."""\n    def __init__(self, root, out, workers, neighbours):\n        paths = [find_sift(root, s) for s in ("train", "test")]\n        print("SIFT sources:", *paths, sep="\\n", flush=True)\n        self.z = [SiftSource(path) for path in paths]\n        self.read_lock = Lock()\n        self.workers, self.neighbours = workers, neighbours\n        fingerprint = hashlib.sha256(b.IDS.tobytes() if b.IDS.dtype != object else "\\n".join(b.IDS).encode())\n        fingerprint.update(np.ascontiguousarray(b.X).tobytes())\n        self.path = out / ("sift_mutual_" + fingerprint.hexdigest()[:16] + ".npz")\n        self.done = {}\n        if self.path.exists():\n            with np.load(self.path) as z:\n                self.done = {tuple(p): tuple(v) for p, v in zip(z["pairs"], z["values"])}\n\n    @lru_cache(maxsize=1200)\n    def descriptors(self, i):\n        iid = b.IDS[i]\n        z = self.z[0 if iid.startswith("tr_") else 1]\n        # NpzFile\'s shared ZIP handle is not safe for concurrent reads.\n        with self.read_lock:\n            d = np.asarray(z[f"{iid}_desc"], dtype=np.float32)\n            xy = np.asarray(z[f"{iid}_xy"], dtype=np.float32)\n        if d.ndim != 2 or xy.shape != (len(d), 2):\n            raise ValueError(f"Malformed SIFT arrays for {iid}")\n        d = np.sqrt(d / np.maximum(d.sum(1, keepdims=True), 1))\n        return d, xy\n\n    def verify(self, pair):\n        a, c = pair\n        da, xa = self.descriptors(a)\n        dc, xc = self.descriptors(c)\n        if min(len(da), len(dc)) < 4:\n            return 0, 0\n        bf = cv2.BFMatcher()\n        back = np.array([m.trainIdx for m in bf.match(dc, da)])\n        matches = bf.knnMatch(da, dc, k=2)\n        good = [m[0] for m in matches if len(m) == 2 and m[0].distance < .85*m[1].distance\n                and back[m[0].trainIdx] == m[0].queryIdx]\n        if len(good) < 4:\n            return len(good), 0\n        # Local RNG seed per task. OpenCV RNG state is thread-local.\n        cv2.setRNGSeed(int((int(a)*100003 + int(c)) % (2**31-1)))\n        _, mask = cv2.estimateAffinePartial2D(\n            np.float32([xa[m.queryIdx] for m in good]),\n            np.float32([xc[m.trainIdx] for m in good]),\n            method=cv2.RANSAC, ransacReprojThreshold=15.)\n        return len(good), int(mask.sum()) if mask is not None else 0\n\n    def activate(self, indices):\n        wanted = set()\n        for sp in ("hyena", "leopard"):\n            ix = indices[b.SP[indices] == sp]\n            # Chunked similarities avoid allocating the full role x role matrix.\n            for start in range(0, len(ix), 256):\n                q = ix[start:start+256]\n                s = b.X[q] @ b.X[ix].T\n                s[np.arange(len(q)), np.arange(start, start+len(q))] = -np.inf\n                k = min(self.neighbours, len(ix)-1)\n                nb = np.argsort(-s, axis=1)[:, :k]\n                for a, row in zip(q, nb):\n                    wanted.update((min(int(a), int(c)), max(int(a), int(c))) for c in ix[row])\n        missing = sorted(wanted - self.done.keys())\n        print(f"SIFT: {len(wanted)} role-local pairs; {len(missing)} uncached", flush=True)\n        with ThreadPoolExecutor(max_workers=self.workers) as pool:\n            for j, (pair, value) in enumerate(zip(missing, pool.map(self.verify, missing)), 1):\n                self.done[pair] = value\n                if j % 5000 == 0:\n                    self.save()\n                    print(f"  verified {j}/{len(missing)}", flush=True)\n        self.save()\n        # Only expose requested pairs, even when cache contains other roles/test.\n        pp = np.array(sorted(wanted), dtype=int).reshape(-1, 2)\n        vv = np.array([self.done[tuple(p)] for p in pp]).reshape(-1, 2)\n        n = len(b.X)\n        def matrix(v):\n            a, c = pp.T\n            return sparse.coo_matrix((np.r_[v, v], (np.r_[a, c], np.r_[c, a])), shape=(n, n)).tocsr()\n        b.GOOD, b.INL = matrix(vv[:, 0]), matrix(vv[:, 1])\n\n    def save(self):\n        """Persist completed verifications atomically so interrupted runs can resume."""\n        pairs = sorted(self.done)\n        tmp = self.path.with_suffix(".tmp.npz")\n        np.savez_compressed(tmp, pairs=np.array(pairs, dtype=np.int64).reshape(-1, 2),\n                            values=np.array([self.done[p] for p in pairs]).reshape(-1, 2))\n        os.replace(tmp, self.path)\n\n\ndef allocate_roles(train, rounds, seed):\n    rng = np.random.default_rng(seed)\n    roles = {f"stage{i+1}": [] for i in range(rounds)} | {"calibration": [], "validation": []}\n    for sp in ("hyena", "leopard"):\n        names = np.array(sorted(train.loc[train.species == sp, "identity"].unique()))\n        rng.shuffle(names)\n        n = max(1, int(round(.2*len(names))))\n        roles["validation"].extend(names[:n])\n        roles["calibration"].extend(names[n:2*n])\n        for i, block in enumerate(np.array_split(names[2*n:], rounds)):\n            roles[f"stage{i+1}"].extend(block)\n    sets = [set(v) for v in roles.values()]\n    assert all(not a & c for i, a in enumerate(sets) for c in sets[i+1:])\n    return {k: np.flatnonzero(train.identity.isin(v).to_numpy()) for k, v in roles.items()}\n\n\ndef split_episode(indices, seed, cap=32, link=(.95, 100)):\n    """Unknown identities absent from gallery; known-query duplicate groups kept whole."""\n    rng = np.random.default_rng(seed)\n    gallery, known, unknown = [], [], []\n    for sp in ("hyena", "leopard"):\n        ix = indices[b.SP[indices] == sp]\n        names = np.unique(b.Y[ix])\n        rng.shuffle(names)\n        nu = max(1, int(round(rng.uniform(.2, .35)*len(names))))\n        unseen = set(names[:nu])\n        for name in names:\n            rows = ix[b.Y[ix] == name].copy()\n            rng.shuffle(rows)\n            if name in unseen:\n                unknown.extend(rows[:cap])\n                continue\n            held = []\n            groups = b.duplicate_groups(rows, link)\n            target = min(max(1, int(round(.35*len(rows)))), cap)\n            for c in rng.permutation(np.unique(groups)):\n                members = rows[groups == c].tolist()\n                if len(held) < target and len(held)+len(members) < len(rows):\n                    held.extend(members)\n            # Never truncate a duplicate group after selecting it.\n            known.extend(held)\n            gallery.extend(i for i in rows if i not in set(held))\n    gi, qi = np.array(gallery, dtype=int), np.array(known+unknown, dtype=int)\n    if not len(known) or not len(unknown):\n        raise ValueError("Episode lacks known/unknown queries; use a larger role or different seed")\n    for sp in ("hyena", "leopard"):\n        if len(np.unique(b.Y[gi[b.SP[gi] == sp]])) < b.K or np.sum(b.SP[qi] == sp) < 6:\n            raise ValueError("Too few identities/queries per species; reduce --rounds or --candidates")\n    assert not set(b.Y[unknown]) & set(b.Y[gi])\n    assert set(b.Y[known]) <= set(b.Y[gi])\n    return gi, qi, np.where(np.isin(qi, unknown), NEW, b.Y[qi])\n\n\ndef frame(gi, qi, truth=None):\n    s = b.build_frames(gi, qi)\n    if truth is not None:\n        lookup = dict(zip(qi, truth))\n        s["truth"] = np.array([lookup[i] for i in s["order"]])\n        s["q"]["unk"] = (s["truth"] == NEW).astype(int)\n        s["c"]["y"] = (s["c"].ident.to_numpy() == s["truth"][s["c"].q.to_numpy()]).astype(int)\n    return s\n\n\ndef through_stack(s, models):\n    for i, m in enumerate(models):\n        r = b.predict(m, s)\n        if i < len(models)-1:\n            s = b.augment(s, r)\n    return s, r\n', 'verifier.py': '"""Full-vector pair MLP and gallery-aware unknown calibration. No raw images."""\nimport copy\nimport numpy as np\nfrom scipy.sparse import csr_matrix\nfrom scipy.sparse.csgraph import connected_components\nfrom sklearn.neural_network import MLPClassifier\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.metrics import log_loss, roc_auc_score\n\n\ndef pair_features(x, a, c):\n    """Symmetric, full 2152-D descriptor interactions, without identity inputs."""\n    return np.c_[np.abs(x[a]-x[c]),x[a]*x[c],np.sum(x[a]*x[c],axis=1)].astype(np.float32)\n\n\ndef sample_pairs(x,y,sp,indices,seed,max_pairs=24000):\n    rng=np.random.default_rng(seed); pairs=[]; labels=[]\n    for species in (\'hyena\',\'leopard\'):\n        ix=indices[sp[indices]==species]; sim=x[ix]@x[ix].T\n        groups=np.zeros(len(ix),int); offset=0\n        for name in np.unique(y[ix]):\n            loc=np.flatnonzero(y[ix]==name)\n            _,gg=connected_components(csr_matrix(sim[np.ix_(loc,loc)]>=.97),directed=False)\n            groups[loc]=gg+offset; offset+=gg.max()+1\n        # Cap anchor appearances per identity so large identities do not dominate.\n        anchors=[]\n        for name in np.unique(y[ix]):\n            loc=np.flatnonzero(y[ix]==name)\n            anchors.extend(rng.choice(loc,min(24,len(loc)),replace=False))\n        for j in anchors:\n            pos=np.flatnonzero((y[ix]==y[ix[j]])&(groups!=groups[j]))\n            neg=np.flatnonzero(y[ix]!=y[ix[j]])\n            if not len(pos) or not len(neg):continue\n            hard=neg[np.argsort(-sim[j,neg])[:30]]\n            for _ in range(2):\n                for candidate,label in ((rng.choice(pos),1),(rng.choice(hard),0),(rng.choice(neg),0)):\n                    pairs.append((ix[j],ix[candidate])); labels.append(label)\n    if not pairs:raise ValueError(\'No usable training pairs\')\n    pairs=np.asarray(pairs); labels=np.asarray(labels)\n    # Deduplicate unordered pairs before sampling to avoid duplicate weights.\n    pairs=np.sort(pairs,axis=1); _,keep=np.unique(pairs,axis=0,return_index=True)\n    keep=rng.permutation(keep)[:max_pairs]\n    if len(np.unique(labels[keep]))!=2:raise ValueError(\'Pair classes missing\')\n    return pairs[keep],labels[keep]\n\n\ndef fit_verifier(x,y,sp,indices,seed=2052,epochs=40,max_pairs=24000):\n    rng=np.random.default_rng(seed); fit=[]; stop=[]\n    for species in (\'hyena\',\'leopard\'):\n        names=np.unique(y[indices[sp[indices]==species]]); rng.shuffle(names)\n        n=max(2,round(.2*len(names)))\n        stop.extend(indices[(sp[indices]==species)&np.isin(y[indices],names[:n])])\n        fit.extend(indices[(sp[indices]==species)&np.isin(y[indices],names[n:])])\n    fit=np.asarray(fit); stop=np.asarray(stop)\n    assert not set(y[fit]) & set(y[stop])\n    fp,fy=sample_pairs(x,y,sp,fit,seed,max_pairs)\n    ep,ey=sample_pairs(x,y,sp,stop,seed+1,min(8000,max_pairs))\n    scaler=StandardScaler().fit(pair_features(x,*fp.T))\n    transform=lambda pairs:np.clip(scaler.transform(pair_features(x,*pairs.T)),-6,6).astype(np.float32)\n    xx,ee=transform(fp),transform(ep)\n    net=MLPClassifier(hidden_layer_sizes=(32,16),alpha=1.,batch_size=256,\n        learning_rate_init=.0003,random_state=seed,early_stopping=False)\n    best=None; best_loss=np.inf; stale=0; history=[]\n    for epoch in range(1,epochs+1):\n        order=rng.permutation(len(fy)); net.partial_fit(xx[order],fy[order],classes=[0,1])\n        pr=net.predict_proba(ee)[:,1]; loss=log_loss(ey,pr,labels=[0,1])\n        history.append(dict(epoch=epoch,stop_logloss=float(loss),stop_auc=float(roc_auc_score(ey,pr))))\n        print(\'MLP epoch\',epoch,\'identity-stop loss\',round(loss,5),flush=True)\n        if loss<best_loss-1e-4:best_loss=loss; best=copy.deepcopy(net); stale=0\n        else:stale+=1\n        if stale>=6:break\n    parameters=sum(v.size for v in best.coefs_+best.intercepts_)\n    return dict(network=best,scaler=scaler,history=history,parameters=parameters,\n                fit_indices=fit,stop_indices=stop,fit_pairs=len(fy),stop_pairs=len(ey))\n\n\ndef pair_scores(model,x,a,c,batch=512):\n    result=[]\n    for start in range(0,len(a),batch):\n        f=pair_features(x,a[start:start+batch],c[start:start+batch])\n        z=np.clip(model[\'scaler\'].transform(f),-6,6)\n        result.append(model[\'network\'].predict_proba(z)[:,1])\n    return np.concatenate(result) if result else np.array([])\n\n\ndef query_features(model,x,y,gallery,s,r):\n    """No query truth is used. Score every gallery photo of all 15 candidates."""\n    n=len(s[\'q\']); k=15; ids=s[\'c\'].ident.to_numpy().reshape(n,k)\n    blocks={name:gallery[y[gallery]==name] for name in np.unique(ids)}\n    aa=[]; cc=[]; sizes=[]\n    for i,query in enumerate(s[\'order\']):\n        for name in ids[i]:\n            g=blocks[name]; aa.extend([query]*len(g)); cc.extend(g); sizes.append(len(g))\n    pp=pair_scores(model,x,np.asarray(aa),np.asarray(cc)); statistics=[]; offset=0\n    for size in sizes:\n        v=np.sort(pp[offset:offset+size])[::-1]; offset+=size\n        statistics.append([v[0],v[:2].mean(),v[:3].mean(),v.mean(),(v>.5).mean()])\n    statistics=np.asarray(statistics).reshape(n,k,5)\n    count=np.asarray(sizes).reshape(n,k); top=r[\'pc\'].argmax(1); ix=np.arange(n)\n    get=lambda key:s[\'c\'][key].to_numpy().reshape(n,k)[ix,top]\n    sorted_pc=np.sort(r[\'pc\'],axis=1)\n    # The control and MLP arm have identical baseline context and calibration model.\n    context=np.c_[r[\'pn\'],r[\'pb\'],np.clip(1-r[\'raw\'].sum(1),0,1),sorted_pc[:,-1],sorted_pc[:,-2],\n        -(r[\'pc\']*np.log(np.maximum(r[\'pc\'],1e-12))).sum(1),np.log1p(count[ix,top]),\n        get(\'mxr\'),get(\'mxl\'),np.log1p(get(\'sx\')),np.log1p(get(\'n_flank\')),\n        np.log1p(get(\'n_view\')),s[\'q\'].sp.to_numpy(),s[\'q\'].view.to_numpy()<0]\n    sorted_best=np.sort(statistics[:,:,0],axis=1)\n    richest=statistics[:,:,0].argmax(1)\n    extra=np.c_[statistics[ix,top],statistics.max(1),\n        (statistics*r[\'pc\'][:,:,None]).sum(1),sorted_best[:,-1]-sorted_best[:,-2],\n        np.log1p(count[ix,richest]),np.log1p(count.sum(1))]\n    return np.nan_to_num(context),np.nan_to_num(np.c_[context,extra])\n\n\ndef calibrator(features,unknown):\n    # Query probabilities are learned on separate calibration identities;\n    # pair scores themselves are never treated as calibrated P(known).\n    return make_pipeline(StandardScaler(),LogisticRegression(C=.1,max_iter=1000)).fit(features,unknown)\n\n\ndef rank(candidates,pc,pnew):\n    labels=np.c_[candidates,np.full(len(pc),\'new_individual\',dtype=object)]\n    scores=np.c_[(1-pnew[:,None])*pc,pnew]\n    return np.take_along_axis(labels,np.argsort(-scores,axis=1)[:,:5],axis=1)\n\n\ndef rr5(pred,truth):\n    return np.array([next((1/(i+1) for i,label in enumerate(row) if label==target),0.) for row,target in zip(pred,truth)])\n'}
    exec(RECOVERY_SOURCE)
    CODE, OUTPUT, INPUT, SAMPLE = prepare(TRAINING_SOURCES, preferred=globals().get('OUTPUT'))
    
    SUBMISSION_SOURCE = '"""Generate competition predictions from the saved Model 11 checkpoints."""\nimport os\nos.environ.setdefault(\'OMP_NUM_THREADS\',\'4\')\nos.environ.setdefault(\'OPENBLAS_NUM_THREADS\',\'4\')\nimport sys\nfrom pathlib import Path\nHERE=Path(__file__).resolve().parent\nsys.path.insert(0,str(HERE.parents[2]/\'tmp/abhay_deps\'))\nimport argparse,json,hashlib,warnings\nimport numpy as np\nimport pandas as pd\nimport joblib,cv2\nfrom threadpoolctl import threadpool_limits\nimport dalvin_core as b\nimport data_protocol as p\nimport verifier as v\nfrom concurrent.futures import ThreadPoolExecutor\nfrom scipy import sparse\n\n\nclass SubmissionSiftCache(p.SiftCache):\n    """Keep original neighbour selection; skip edges between two gallery photos."""\n    def activate(self,indices):\n        wanted=set()\n        for species in (\'hyena\',\'leopard\'):\n            ix=indices[b.SP[indices]==species]\n            for start in range(0,len(ix),256):\n                queries=ix[start:start+256]\n                sim=b.X[queries]@b.X[ix].T\n                sim[np.arange(len(queries)),np.arange(start,start+len(queries))]=-np.inf\n                nb=np.argsort(-sim,axis=1)[:,:min(self.neighbours,len(ix)-1)]\n                for a,row in zip(queries,nb):\n                    for c in ix[row]:\n                        if int(a) in self.query_indices or int(c) in self.query_indices:\n                            wanted.add(tuple(sorted((int(a),int(c)))))\n        missing=sorted(wanted-self.done.keys())\n        print(\'Submission SIFT:\',len(wanted),\'required pairs;\',len(missing),\'uncached; gallery/gallery checks skipped\',flush=True)\n        with ThreadPoolExecutor(max_workers=self.workers) as pool:\n            for j,(key,value) in enumerate(zip(missing,pool.map(self.verify,missing)),1):\n                self.done[key]=value\n                if j%5000==0:self.save();print(\'Verified\',j,\'/\',len(missing),flush=True)\n        self.save()\n        keys=np.array(sorted(wanted),dtype=int).reshape(-1,2)\n        values=np.array([self.done[tuple(k)] for k in keys]).reshape(-1,2)\n        def matrix(column):\n            a,c=keys.T;v=values[:,column]\n            return sparse.coo_matrix((np.r_[v,v],(np.r_[a,c],np.r_[c,a])),shape=(len(b.X),len(b.X))).tocsr()\n        b.GOOD,b.INL=matrix(0),matrix(1)\n\n\ndef submission_frame(sample,test,train,order_ids,pred):\n    if not sample.id.is_unique or set(sample.id)!=set(test.id):\n        raise ValueError(\'Sample submission IDs do not match test.csv\')\n    frame=pd.DataFrame({\'id\':order_ids,\'predictions\':[\' \'.join(row) for row in pred]})\n    if not frame.id.is_unique or set(frame.id)!=set(test.id):\n        raise ValueError(\'Predictions do not cover every test ID exactly once\')\n    result=sample[[\'id\']].merge(frame,on=\'id\',how=\'left\',validate=\'one_to_one\')\n    species=test.set_index(\'id\').species\n    allowed=train.groupby(\'species\').identity.apply(set).to_dict()\n    for row in result.itertuples(index=False):\n        labels=row.predictions.split()\n        if len(labels)!=5 or len(set(labels))!=5:raise ValueError(\'Need five distinct labels\')\n        if any(label!=\'new_individual\' and label not in allowed[species[row.id]] for label in labels):\n            raise ValueError(\'Invalid identity or cross-species prediction\')\n    assert result.id.tolist()==sample.id.tolist()\n    return result\n\n\ndef main():\n    ap=argparse.ArgumentParser()\n    ap.add_argument(\'--data\',type=Path,required=True)\n    ap.add_argument(\'--sample\',type=Path,required=True)\n    ap.add_argument(\'--run\',type=Path,required=True)\n    ap.add_argument(\'--workers\',type=int,default=4)\n    args=ap.parse_args();run=args.run\n    config=json.loads((run/\'config.json\').read_text())\n    for name in (\'verifier.py\',\'dalvin_core.py\',\'data_protocol.py\'):\n        if hashlib.sha256((HERE/name).read_bytes()).hexdigest()!=config[\'sources\'][name]:\n            raise ValueError(f\'{name} changed since training; restore the training source snapshot\')\n    models=joblib.load(run/\'baseline_models.joblib\')\n    if len(models)!=3:raise ValueError(\'Baseline training has not completed all three stages\')\n    network=joblib.load(run/\'verifier.joblib\')\n    calibrators=joblib.load(run/\'calibrators.joblib\')\n    cv2.setNumThreads(1)\n    warnings.filterwarnings(\'ignore\',message=\'Only one sample available.*\',category=UserWarning)\n    train,test,meta=p.load_data(args.data);b.NX=b.X\n    sample=pd.read_csv(args.sample)\n    if not sample.id.is_unique or set(sample.id)!=set(test.id):raise ValueError(\'Wrong sample submission file\')\n    # All labelled photos are now gallery; every test photo is an unlabelled query.\n    # Role-local caches are reselected for this full population by activate().\n    cache=SubmissionSiftCache(args.data,run,args.workers,150)\n    cache.query_indices=set(range(len(train),len(meta)))\n    print(\'Preparing full-gallery SIFT. This can take substantially longer than validation.\',flush=True)\n    cache.activate(np.arange(len(meta)))\n    gi=np.arange(len(train));qi=np.arange(len(train),len(meta))\n    s=p.frame(gi,qi);s,r=p.through_stack(s,models)\n    print(\'Computing verifier evidence against full labelled gallery\',flush=True)\n    _,features=v.query_features(network,b.X,b.Y,gi,s,r)\n    pnew=.5*(r[\'pb\']+calibrators[\'mlp\'].predict_proba(features)[:,1])\n    candidates=s[\'c\'].ident.to_numpy().reshape(-1,15)\n    for name,score in [(\'mlp_blend\',pnew),(\'baseline\',r[\'pb\'])]:\n        pred=v.rank(candidates,r[\'pc\'],score)\n        result=submission_frame(sample,test,train,b.IDS[s[\'order\']],pred)\n        dest=run/f\'submission_{name}.csv\'\n        result.to_csv(dest,index=False)\n        print(\'WROTE\',dest,\'rows\',len(result),flush=True)\n    (run/\'submission_info.json\').write_text(json.dumps(dict(\n        gallery_photos=len(train),test_queries=len(test),method=\'fixed 50/50 MLP blend\',\n        checkpoints=\'Frozen baseline, verifier and calibrators from validation experiment; no refitting\',\n        caveat=\'Full-gallery distribution differs from calibration/validation. No leaderboard improvement is established.\'),indent=2))\n\n\nif __name__==\'__main__\':\n    with threadpool_limits(limits=4,user_api=\'blas\'):main()\n'
    
    PERSISTENCE_SOURCE = '"""Portable, checksummed Model 11 checkpoint bundles; never includes input data."""\nfrom pathlib import Path,PurePosixPath\nimport json,hashlib,zipfile,os,platform,tempfile,shutil\nREQUIRED=(\'config.json\',\'baseline_models.joblib\',\'verifier.joblib\',\'calibrators.joblib\')\n\ndef digest(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda:stream.read(1024*1024),b\'\'):h.update(block)\n    return h.hexdigest()\n\ndef save_bundle(run,code,archive):\n    run,code,archive=map(Path,(run,code,archive));archive.parent.mkdir(parents=True,exist_ok=True)\n    if (run/\'config.json\').exists():\n        config=json.loads((run/\'config.json\').read_text())\n        for name in (\'dalvin_core.py\',\'data_protocol.py\',\'verifier.py\'):\n            expected=config.get(\'sources\',{}).get(name)\n            if expected and (not (code/name).exists() or digest(code/name)!=expected):\n                raise ValueError(f\'Backup source does not match the trained checkpoint: {name}\')\n    files=[]\n    for label,folder in [(\'run\',run),(\'code\',code)]:\n        if folder.exists():\n            for path in sorted(folder.iterdir()):\n                if path.is_file() and path.suffix not in (\'.tmp\',\'.zip\',\'.pyc\'):\n                    files.append((f\'{label}/{path.name}\',path))\n    if not files:raise FileNotFoundError(\'No checkpoints or source files to save\')\n    manifest=dict(format=1,python=platform.python_version(),\n        complete_for_submission=all((run/name).is_file() for name in REQUIRED),\n        missing_for_submission=[name for name in REQUIRED if not (run/name).is_file()],files={})\n    temp=archive.with_suffix(\'.tmp\')\n    try:\n        with zipfile.ZipFile(temp,\'w\',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as z:\n            for name,path in files:\n                before=digest(path);z.write(path,name)\n                if digest(path)!=before:raise RuntimeError(f\'File changed during backup: {path}; retry after computation stops\')\n                manifest[\'files\'][name]=before\n            z.writestr(\'manifest.json\',json.dumps(manifest,indent=2))\n        # Verify the actual archive contents before publishing the backup.\n        with zipfile.ZipFile(temp) as z:\n            for name,expected in manifest[\'files\'].items():\n                if hashlib.sha256(z.read(name)).hexdigest()!=expected:raise ValueError(\'Backup checksum mismatch\')\n        os.replace(temp,archive)\n    finally:\n        if temp.exists():temp.unlink()\n    print(\'Saved checkpoint bundle:\',archive,\'complete for submission:\',manifest[\'complete_for_submission\'])\n    return manifest\n\ndef restore_bundle(archive,run,code):\n    run,code=Path(run),Path(code)\n    with zipfile.ZipFile(archive) as z:\n        manifest=json.loads(z.read(\'manifest.json\'))\n        if manifest.get(\'format\')!=1:raise ValueError(\'Unknown checkpoint bundle format\')\n        if len(z.namelist())!=len(set(z.namelist())):raise ValueError(\'Duplicate archive members\')\n        if set(z.namelist())!=set(manifest[\'files\'])|{\'manifest.json\'}:raise ValueError(\'Unexpected archive members\')\n        for name,expected in manifest[\'files\'].items():\n            parts=PurePosixPath(name).parts\n            if len(parts)!=2 or parts[0] not in (\'run\',\'code\') or parts[1] in (\'.\',\'..\') or \'\\\\\' in name or \':\' in name:\n                raise ValueError(\'Unsafe archive path\')\n            if hashlib.sha256(z.read(name)).hexdigest()!=expected:raise ValueError(f\'Corrupt bundle: {name}\')\n            destination=(run if parts[0]==\'run\' else code)/parts[1]\n            if destination.exists() and digest(destination)!=expected:\n                raise ValueError(f\'Restore would overwrite different files at {destination}. Use empty OUTPUT/CODE folders.\')\n        # All members and existing destinations are verified before writing.\n        for name in manifest[\'files\']:\n            group,filename=PurePosixPath(name).parts\n            destination=(run if group==\'run\' else code)/filename\n            destination.parent.mkdir(parents=True,exist_ok=True)\n            destination.write_bytes(z.read(name))\n    print(\'Restored verified checkpoints and sources:\',run)\n    return manifest\n'
    (CODE/'predict_submission.py').write_text(SUBMISSION_SOURCE, encoding='utf-8')
    exec(PERSISTENCE_SOURCE)
    try:
        subprocess.run([sys.executable, '-u', str(CODE/'predict_submission.py'),
            '--data', str(INPUT), '--sample', str(SAMPLE), '--run', str(OUTPUT),
            '--workers', '4'], check=True)
    finally:
        backup_code = Path(CHECKPOINT_CODE) if CHECKPOINT_CODE and (Path(CHECKPOINT_CODE)/'run.py').exists() else CODE
        bundle = Path('/kaggle/working/model11_checkpoint_bundle.zip')
        save_bundle(OUTPUT, backup_code, bundle)
        display(FileLink(str(bundle)))
    display(FileLink(str(OUTPUT/'submission_mlp_blend.csv')))
    display(FileLink(str(OUTPUT/'submission_baseline.csv')))
else:
    print("Submission skipped. Set GENERATE_SUBMISSION=True or MODE=\"submission\" to generate it.")
